# Spectral Joint Flow (SJFPO) on Kaggle (offline)

Train the spectral flow policy on MuJoCo with internet disabled. The cells
below recreate the full source tree (`src/`, `scripts/train.py`, `configs/`)
next to this notebook via `%%writefile`, install deps from the attached
wheels dataset, then run `scripts/train.py` (eval runs inside training and
lands in `eval_history.csv`).

## 0. Attachments and secrets

1. Attach your wheels dataset (the pinned `requirements-linux.txt` bundle)
   and set `WHEELS_DIR` in the install cell to its `/kaggle/input/...` path.
2. Set the `WANDB_API_KEY` secret so wandb login happens automatically
   (no manual paste). Training also runs without it.
3. Run all cells top to bottom.

In [ ]:
# Recreate the source tree next to the notebook (writefile needs the dirs).
!mkdir -p src/sjfpo scripts configs

In [ ]:
# Detect runtime and load secrets from the right store.
import os

def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

In [ ]:
%%writefile src/sjfpo/__init__.py

In [ ]:
%%writefile src/sjfpo/analysis.py
"""Benchmark analysis: bootstrap CIs, IQM, superiority, normalization, plots, CSV."""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")  # Non-interactive backend for tests
import matplotlib.pyplot as plt
import numpy as np

from sjfpo.config import RunSpec


class AnalysisError(Exception):
    """Analysis cannot proceed (missing/invalid data)."""

    pass


def bootstrap_ci(
    data: np.ndarray,
    n_boot: int = 2000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple[float, float, float]:
    if len(data) == 0:
        raise ValueError("data must not be empty")

    rng = np.random.default_rng(seed)
    n = len(data)
    boot_means = np.empty(n_boot)

    for i in range(n_boot):
        indices = rng.integers(0, n, size=n)
        boot_sample = data[indices]
        boot_means[i] = np.mean(boot_sample)

    alpha = (1 - ci) / 2
    ci_low = np.percentile(boot_means, 100 * alpha)
    ci_high = np.percentile(boot_means, 100 * (1 - alpha))
    mean = np.mean(data)

    return float(mean), float(ci_low), float(ci_high)


def compute_iqm(returns_per_seed: list[float]) -> float:
    if not returns_per_seed:
        raise ValueError("returns_per_seed must not be empty")

    arr = np.array(returns_per_seed, dtype=float)
    n = len(arr)

    if n < 4:
        return float(np.mean(arr))

    sorted_arr = np.sort(arr)
    trim_count = n // 4

    if trim_count == 0:
        return float(np.mean(arr))

    trimmed = sorted_arr[trim_count : n - trim_count]
    return float(np.mean(trimmed))


def is_superior(ci_low: float, ci_high: float) -> bool:
    return ci_low > 0


def normalize_cross_env(
    results_by_env: dict[str, dict[str, list[float]]],
) -> dict[str, dict[str, list[float]]]:
    normalized = {}

    for env_name, methods in results_by_env.items():
        all_final_returns = []
        for method_returns in methods.values():
            if method_returns:
                all_final_returns.append(method_returns[-1])

        if not all_final_returns:
            normalized[env_name] = {m: [] for m in methods}
            continue

        max_return = max(all_final_returns)
        if max_return == 0:
            max_return = 1.0

        normalized[env_name] = {}
        for method, returns in methods.items():
            normalized[env_name][method] = [r / max_return for r in returns]

    return normalized


def load_eval_histories(run_dir: Path, *, eval_steps: int) -> list[dict[str, Any]]:
    """Scan run dirs for eval_history.csv files.

    Returns [{method, env, seed, global_step, return}]; global_step is
    episode_index * eval_steps (the CSV has no step column).
    Malformed rows/headers raise loudly, never default to zero.
    """
    histories = []

    for subdir in run_dir.iterdir():
        if not subdir.is_dir():
            continue

        parts = subdir.name.split("_")
        if len(parts) < 3:
            continue

        method = parts[0]
        env = "_".join(parts[1:-1])
        seed_str = parts[-1].removeprefix("seed")
        try:
            seed = int(seed_str)
        except ValueError:
            continue

        csv_path = subdir / "eval_history.csv"
        if not csv_path.exists():
            continue

        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            missing = {"episode", "return"} - set(reader.fieldnames or [])
            if missing:
                raise AnalysisError(f"{csv_path} missing columns: {sorted(missing)}")
            for row in reader:
                histories.append(
                    {
                        "method": method,
                        "env": env,
                        "seed": seed,
                        "global_step": int(row["episode"]) * eval_steps,
                        "return": float(row["return"]),
                    }
                )

    return histories


@dataclass(frozen=True)
class AnalysisReport:

    per_env_curves: dict[str, Path]
    aggregate_png: Path
    summary_csv: Path
    superiority: dict[str, bool]
    excluded_diverged: list[str]
    summary_stats: dict[tuple[str, str], tuple[float, float, float]] = field(
        default_factory=dict  # (method, env) -> (iqm, ci_low, ci_high)
    )


def plot_per_env_curves(
    env_name: str,
    curves_by_method: dict[str, dict[str, Any]],
    output_dir: Path,
) -> Path:
    """Learning curves with bootstrap-CI shading for one env."""
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {"sjfpo": "#1f77b4", "ppo": "#ff7f0e", "sac": "#2ca02c"}

    for method, data in curves_by_method.items():
        eval_points = data["eval_points"]
        returns_per_seed = data["returns_per_seed"]

        if not returns_per_seed:
            continue

        returns_array = np.array(returns_per_seed)
        mean_returns = np.mean(returns_array, axis=0)

        ci_low = np.empty(len(eval_points))
        ci_high = np.empty(len(eval_points))
        for i in range(len(eval_points)):
            point_returns = returns_array[:, i]
            if len(point_returns) >= 2:
                _, low, high = bootstrap_ci(point_returns, n_boot=500, ci=0.95, seed=42)
                ci_low[i] = low
                ci_high[i] = high
            else:
                ci_low[i] = mean_returns[i]
                ci_high[i] = mean_returns[i]

        color = colors.get(method, "#000000")
        ax.plot(
            eval_points, mean_returns, label=method.upper(), color=color, linewidth=2
        )
        ax.fill_between(eval_points, ci_low, ci_high, color=color, alpha=0.2)

    ax.set_xlabel("Global Step")
    ax.set_ylabel("Return")
    ax.set_title(f"Learning Curves: {env_name}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))

    plt.tight_layout()
    png_path = output_dir / f"{env_name}_curves.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return png_path


def plot_aggregate(
    aggregate_data: dict[str, dict[str, list[float]]],
    output_dir: Path,
) -> Path:
    """Normalized cross-env bar chart with bootstrap-CI error bars."""
    output_dir.mkdir(parents=True, exist_ok=True)

    methods = set()
    for env_data in aggregate_data.values():
        methods.update(env_data.keys())
    methods = sorted(methods)

    method_means = {}
    method_cis = {}

    for method in methods:
        all_returns = []
        for env_data in aggregate_data.values():
            if method in env_data:
                all_returns.extend(env_data[method])

        if all_returns:
            mean, ci_low, ci_high = bootstrap_ci(
                np.array(all_returns), n_boot=2000, ci=0.95, seed=42
            )
            method_means[method] = mean
            method_cis[method] = (ci_low, ci_high)
        else:
            method_means[method] = 0.0
            method_cis[method] = (0.0, 0.0)

    fig, ax = plt.subplots(figsize=(8, 6))

    colors = {"sjfpo": "#1f77b4", "ppo": "#ff7f0e", "sac": "#2ca02c"}
    x_pos = np.arange(len(methods))
    means = [method_means[m] for m in methods]
    ci_lows = [method_cis[m][0] for m in methods]
    ci_highs = [method_cis[m][1] for m in methods]

    bars = ax.bar(
        x_pos,
        means,
        yerr=[
            np.array(means) - np.array(ci_lows),
            np.array(ci_highs) - np.array(means),
        ],
        capsize=5,
        color=[colors.get(m, "#000000") for m in methods],
        edgecolor="black",
        linewidth=0.5,
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels([m.upper() for m in methods])
    ax.set_ylabel("Normalized Return")
    ax.set_title("Aggregate Cross-Environment Comparison")
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    png_path = output_dir / "aggregate_comparison.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return png_path


def write_summary_csv(report: AnalysisReport, output_dir: Path) -> None:
    """Summary CSV (method,env,iqm,ci_low,ci_high,status) from report cells."""
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / "results_summary.csv"

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["method", "env", "iqm", "ci_low", "ci_high", "status"])
        for (method, env), (iqm, ci_low, ci_high) in sorted(
            report.summary_stats.items()
        ):
            status = "complete"
            if any(
                d.startswith(f"{method}_{env}_seed") for d in report.excluded_diverged
            ):
                status = "diverged"
            writer.writerow([method, env, iqm, ci_low, ci_high, status])


def analyze_results(
    matrix: tuple[RunSpec, ...],
    base_dir: Path,
    output_dir: Path,
    *,
    eval_steps: int,
) -> AnalysisReport:
    """Run the full analysis: histories → curves → aggregate → plots → CSV."""
    output_dir.mkdir(parents=True, exist_ok=True)

    histories = load_eval_histories(base_dir, eval_steps=eval_steps)

    if not histories:
        raise AnalysisError("no eval histories")

    from collections import defaultdict

    grouped = defaultdict(lambda: defaultdict(list))

    for h in histories:
        key = (h["method"], h["env"], h["seed"])
        grouped[key]["returns"].append(h["return"])
        grouped[key]["eval_points"].append(h["global_step"])

    excluded_diverged = []
    for subdir in base_dir.iterdir():
        if not subdir.is_dir():
            continue
        manifest_path = subdir / "diverged.json"
        if manifest_path.exists():
            with open(manifest_path) as f:
                manifest = json.load(f)
            if manifest.get("diverged", False):
                excluded_diverged.append(subdir.name)

    per_env_curves = {}
    curves_by_env_method = defaultdict(lambda: defaultdict(list))

    for (method, env, seed), data in grouped.items():
        if f"{method}_{env}_{seed}" in excluded_diverged:
            continue
        curves_by_env_method[env][method].append(
            {
                "seed": seed,
                "eval_points": data["eval_points"],
                "returns": data["returns"],
            }
        )

    for env, methods in curves_by_env_method.items():
        curves_by_method = {}
        for method, seeds_data in methods.items():
            if seeds_data:
                curves_by_method[method] = {
                    "eval_points": seeds_data[0]["eval_points"],
                    "returns_per_seed": [d["returns"] for d in seeds_data],
                }
        if curves_by_method:
            png_path = plot_per_env_curves(env, curves_by_method, output_dir)
            per_env_curves[env] = png_path

    final_returns_by_env = defaultdict(lambda: defaultdict(list))
    for (method, env, seed), data in grouped.items():
        if f"{method}_{env}_{seed}" in excluded_diverged:
            continue
        if data["returns"]:
            final_returns_by_env[env][method].append(data["returns"][-1])

    summary_stats: dict[tuple[str, str], tuple[float, float, float]] = {}
    for env, methods in final_returns_by_env.items():
        for method, returns in methods.items():
            _, ci_low, ci_high = bootstrap_ci(
                np.array(returns, dtype=float), n_boot=2000, ci=0.95, seed=42
            )
            summary_stats[(method, env)] = (compute_iqm(returns), ci_low, ci_high)

    normalized = normalize_cross_env(final_returns_by_env)

    aggregate_png = plot_aggregate(normalized, output_dir)

    superiority = {}
    for method in ["sjfpo", "ppo", "sac"]:
        baseline_methods = [m for m in ["sjfpo", "ppo", "sac"] if m != method]
        if not baseline_methods:
            superiority[method] = False
            continue

        method_returns = []
        for env_data in final_returns_by_env.values():
            if method in env_data:
                method_returns.extend(env_data[method])

        if not method_returns:
            superiority[method] = False
            continue

        best_baseline_returns = []
        for baseline in baseline_methods:
            baseline_returns = []
            for env_data in final_returns_by_env.values():
                if baseline in env_data:
                    baseline_returns.extend(env_data[baseline])
            if baseline_returns:
                best_baseline_returns.append((np.mean(baseline_returns), baseline))

        if not best_baseline_returns:
            superiority[method] = False
            continue

        best_baseline_returns.sort(reverse=True)
        best_baseline_method = best_baseline_returns[0][1]

        baseline_returns = []
        for env_data in final_returns_by_env.values():
            if best_baseline_method in env_data:
                baseline_returns.extend(env_data[best_baseline_method])

        if not baseline_returns:
            superiority[method] = False
            continue

        min_len = min(len(method_returns), len(baseline_returns))
        if min_len < 2:
            superiority[method] = False
            continue

        diff = np.array(method_returns[:min_len]) - np.array(baseline_returns[:min_len])
        _, ci_low, ci_high = bootstrap_ci(diff, n_boot=2000, ci=0.95, seed=42)
        superiority[method] = is_superior(ci_low, ci_high)

    summary_csv = output_dir / "results_summary.csv"
    write_summary_csv(
        AnalysisReport(
            per_env_curves=per_env_curves,
            aggregate_png=aggregate_png,
            summary_csv=summary_csv,
            superiority=superiority,
            excluded_diverged=excluded_diverged,
        ),
        output_dir,
    )

    return AnalysisReport(
        per_env_curves=per_env_curves,
        aggregate_png=aggregate_png,
        summary_csv=summary_csv,
        superiority=superiority,
        excluded_diverged=excluded_diverged,
        summary_stats=summary_stats,
    )

In [ ]:
%%writefile src/sjfpo/baselines.py
"""skrl PPO/SAC baselines sharing Evaluator and checkpointing.

Frame-stacking matches SJFPO (n_obs_steps=16 → flat [obs_dim*16]). No
spectral/flow code here.
"""

from __future__ import annotations

import dataclasses
import json
import time
from collections import deque
from pathlib import Path
from typing import Any

import gymnasium as gym
import numpy as np
import einx
import torch
import torch.nn as nn

from sjfpo.config import PPOConfig, SACConfig
from sjfpo.evaluator import Evaluator, check_divergence, compute_random_baseline
from lerobot.utils.random_utils import set_seed

from sjfpo.utils import maybe_compile, resolve_device, to_numpy_1d
from sjfpo.wandb_logging import (
    init_wandb_run_for_method,
    log_eval_summary,
    log_metrics,
    finish_run,
)

# skrl imports (lazy to allow partial import when skrl is unavailable)

skrl_imported = False
PPO_CFG_cls = None
SAC_CFG_cls = None
PPO_cls = None
SAC_cls = None
RandomMemory_cls = None
wrap_env_fn = None
RunningStandardScaler_cls = None

PPOPolicy = None
PPOValue = None
SACPolicy = None
SACCritic = None


# Provenance: "default" = skrl default; "paper" = FPO++ paper;
# "adaptation" = changed for benchmark fairness.
PROVENANCE_TABLE: list[tuple[str, str, str, str]] = [
    # (method, key, value_source, note)
    ("PPO", "rollouts", "2048", "default"),
    ("PPO", "learning_epochs", "10", "default"),
    ("PPO", "mini_batches", "8", "default"),
    ("PPO", "learning_rate", "3e-4", "paper"),
    ("PPO", "kl_threshold", "0.01", "default"),
    ("PPO", "grad_norm_clip", "1.0", "default"),
    ("PPO", "discount_factor", "0.99", "adaptation"),
    ("PPO", "gae_lambda", "0.95", "adaptation"),
    ("PPO", "ratio_clip", "0.2", "default"),
    ("SAC", "learning_rate", "3e-4", "default"),
    ("SAC", "batch_size", "256", "default"),
    ("SAC", "memory_size", "300000", "adaptation"),
    ("SAC", "gradient_steps", "1", "default"),
    ("SAC", "polyak", "0.005", "default"),
    ("SAC", "initial_random_steps", "5000", "adaptation"),
]


def build_mlp_layer(
    in_dim: int, hidden: tuple[int, ...], out_dim: int
) -> nn.Sequential:
    from sjfpo.networks import SwiGLU

    layers: list[nn.Module] = []
    prev = in_dim
    for h in hidden:
        layers.append(SwiGLU(prev, h, h))
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)


def import_skrl_agent_classes():
    from skrl.agents.torch.ppo import PPO, PPO_CFG
    from skrl.agents.torch.sac import SAC, SAC_CFG
    from skrl.memories.torch import RandomMemory
    from skrl.envs.wrappers.torch import wrap_env
    from skrl.resources.preprocessors.torch import RunningStandardScaler

    return (
        PPO_CFG,
        SAC_CFG,
        PPO,
        SAC,
        RandomMemory,
        wrap_env,
        RunningStandardScaler,
    )


def import_skrl_model_mixins():
    from skrl.models.torch import GaussianMixin as GM, DeterministicMixin as DM, Model

    return GM, DM, Model


def create_gaussian_policy_class(GM, Model):
    """Shared Gaussian actor: MLP → mean + learnable log_std (PPO and SAC)."""

    class GaussianPolicyImpl(GM, Model):
        def __init__(self, obs_dim: int, action_dim: int, hidden: tuple[int, ...]):
            Model.__init__(
                self,
                observation_space=gym.spaces.Box(-1, 1, (obs_dim,)),
                action_space=gym.spaces.Box(-1, 1, (action_dim,)),
                device=str(resolve_device()),
            )
            GM.__init__(
                self,
                clip_actions=True,
                clip_log_std=True,
                min_log_std=-20,
                max_log_std=2,
                reduction="sum",
            )
            self.net = build_mlp_layer(obs_dim, hidden, action_dim)
            self.log_std_parameter = nn.Parameter(torch.zeros(action_dim))

        def compute(self, inputs: dict, role: str = "policy"):
            return self.net(inputs["observations"]), {"log_std": self.log_std_parameter}

    return GaussianPolicyImpl


def create_ppo_value_class(DM, Model):
    """Deterministic value function for PPO: MLP → scalar."""

    class PpoValueImpl(DM, Model):
        def __init__(self, obs_dim: int, hidden: tuple[int, ...]):
            Model.__init__(
                self,
                observation_space=gym.spaces.Box(-1, 1, (obs_dim,)),
                device=str(resolve_device()),
            )
            DM.__init__(self, clip_actions=False)
            self.net = build_mlp_layer(obs_dim, hidden, 1)

        def compute(self, inputs: dict, role: str = "value"):
            return self.net(inputs["observations"]), {}

    return PpoValueImpl


def create_sac_critic_class(DM, Model):
    """Q-critic for SAC: MLP over [obs; action] → scalar."""

    class SacCriticImpl(DM, Model):  # type: ignore[name-defined]
        def __init__(self, obs_dim: int, action_dim: int, hidden: tuple[int, ...]):
            Model.__init__(
                self,
                observation_space=gym.spaces.Box(-1, 1, (obs_dim,)),
                action_space=gym.spaces.Box(-1, 1, (action_dim,)),
                device=str(resolve_device()),
            )
            DM.__init__(self, clip_actions=False)
            self.net = build_mlp_layer(obs_dim + action_dim, hidden, 1)

        def compute(self, inputs: dict, role: str = "critic"):
            x = einx.id(
                "b o, b a -> b (o + a)", inputs["observations"], inputs["taken_actions"]
            )
            return self.net(x), {}

    return SacCriticImpl


def import_skrl_dependencies():
    global skrl_imported, PPO_CFG_cls, SAC_CFG_cls, PPO_cls, SAC_cls
    global RandomMemory_cls, wrap_env_fn, RunningStandardScaler_cls
    global PPOPolicy, PPOValue, SACPolicy, SACCritic

    if skrl_imported:
        return
    (
        PPO_CFG_cls,
        SAC_CFG_cls,
        PPO_cls,
        SAC_cls,
        RandomMemory_cls,
        wrap_env_fn,
        RunningStandardScaler_cls,
    ) = import_skrl_agent_classes()
    GM, DM, Model = import_skrl_model_mixins()
    PPOPolicy = create_gaussian_policy_class(GM, Model)
    PPOValue = create_ppo_value_class(DM, Model)
    SACPolicy = PPOPolicy
    SACCritic = create_sac_critic_class(DM, Model)
    skrl_imported = True


def seed_and_wrap_env(
    env: Any, seed: int, n_obs_steps: int
) -> tuple[Any, Any, int, int]:
    set_seed(seed)
    wrapped_env = FrameStackWrapper(env, n_obs_steps=n_obs_steps)
    skrl_env = wrap_env_fn(wrapped_env, wrapper="gymnasium", verbose=False)
    obs_dim = wrapped_env.observation_space.shape[0]
    action_dim = wrapped_env.action_space.shape[0]
    return wrapped_env, skrl_env, obs_dim, action_dim


def mean_action_eval_policy(agent: Any) -> Any:

    def eval_policy(obs):
        obs_t = einx.id("d -> 1 d", torch.as_tensor(obs, dtype=torch.float32))
        with torch.inference_mode():
            agent.enable_training_mode(False)
            actions, outputs = agent.act(obs_t, None, timestep=0, timesteps=1)
            agent.enable_training_mode(True)
        return to_numpy_1d(einx.id("1 a -> a", outputs.get("mean_actions", actions)))

    return eval_policy

    # Frame-stack wrapper (identical to SJFPO's n_obs_steps treatment)


class FrameStackWrapper(gym.ObservationWrapper):
    """Stack the last n_obs_steps obs into one flat vector for skrl MLPs."""

    def __init__(self, env: gym.Env, n_obs_steps: int = 16):
        super().__init__(env)
        self.n_obs_steps = n_obs_steps
        obs_dim = int(np.prod(env.observation_space.shape))
        self.obs_dim = obs_dim
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim * n_obs_steps,),
            dtype=np.float32,
        )
        self.frames: deque = deque(maxlen=n_obs_steps)

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        self.frames.clear()
        flat = np.asarray(obs, dtype=np.float32).ravel()
        for _ in range(self.n_obs_steps):
            self.frames.append(flat.copy())
        return self.get_current_observation(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        flat = np.asarray(obs, dtype=np.float32).ravel()
        self.frames.append(flat.copy())
        return self.get_current_observation(), reward, terminated, truncated, info

    def get_current_observation(self) -> np.ndarray:
        return np.concatenate(list(self.frames), axis=0).astype(np.float32)


@dataclasses.dataclass
class BaselineLoopState:

    t_start: float
    global_step: int = 0
    eval_count: int = 0
    consecutive_below: int = 0
    diverged: bool = False
    wandb_enabled: bool = False
    env_name: str = "unknown"
    all_eval_returns: list = dataclasses.field(default_factory=list)
    artifacts: list = dataclasses.field(default_factory=list)


def init_baseline_run(env, eval_env, out_dir, method_name, seed, total_steps, cfg_dict):
    out_dir.mkdir(parents=True, exist_ok=True)
    evaluator = Evaluator(eval_env, eval_seed=10000 + seed, max_episodes=12)
    env_unwrapped = getattr(env, "unwrapped", env)
    env_spec = getattr(env_unwrapped, "spec", None)
    env_name_str = (
        getattr(env_spec, "id", "unknown") if env_spec is not None else "unknown"
    )
    wandb_enabled: bool = init_wandb_run_for_method(
        method=method_name,
        env_name=env_name_str,
        seed=seed,
        total_steps=total_steps,
        cfg_dict=cfg_dict or {},
    )
    state = BaselineLoopState(
        t_start=time.time(), wandb_enabled=wandb_enabled, env_name=env_name_str
    )
    return evaluator, state


def resolve_eval_policy(agent, eval_policy_fn):
    if eval_policy_fn is not None:
        return eval_policy_fn
    return lambda obs: (
        agent.act(
            einx.id("d -> 1 d", torch.as_tensor(obs, dtype=torch.float32)),
            None,
            timestep=0,
            timesteps=1,
        )[0]
        .squeeze(0)
        .cpu()
        .numpy()
    )


def query_agent_actions(agent, observations, states, global_step, total_steps):
    with torch.inference_mode():
        actions, _ = agent.act(
            observations
            if torch.is_tensor(observations)
            else torch.as_tensor(observations, dtype=torch.float32),
            states,
            timestep=global_step,
            timesteps=total_steps,
        )
    return actions


def apply_env_step(env, actions):
    next_obs, reward, terminated, truncated, info = env.step(
        actions.float()
        if torch.is_tensor(actions)
        else torch.as_tensor(actions, dtype=torch.float32)
    )
    next_states = env.state() if hasattr(env, "state") else None
    return next_obs, reward, terminated, truncated, next_states


def record_agent_transition(
    agent,
    observations,
    states,
    actions,
    next_obs,
    reward,
    terminated,
    truncated,
    next_states,
    global_step,
    total_steps,
):
    next_obs_t = torch.as_tensor(next_obs, dtype=torch.float32)
    # skrl convention is [num_envs, 1]; [1] breaks in-place bootstrap math.
    reward_t = torch.as_tensor([[reward]], dtype=torch.float32)
    terminated_t = torch.as_tensor([[terminated]])
    truncated_t = torch.as_tensor([[truncated]])
    obs_t = (
        observations
        if torch.is_tensor(observations)
        else torch.as_tensor(observations, dtype=torch.float32)
    )
    agent.record_transition(
        observations=einx.id("d -> 1 d", obs_t) if obs_t.dim() == 1 else obs_t,
        states=states,
        actions=einx.id("a -> 1 a", actions) if actions.dim() == 1 else actions,
        rewards=reward_t,
        next_observations=einx.id("d -> 1 d", next_obs_t)
        if next_obs_t.dim() == 1
        else next_obs_t,
        next_states=next_states,
        terminated=terminated_t,
        truncated=truncated_t,
        infos=None,
        timestep=global_step,
        timesteps=total_steps,
    )
    agent.post_interaction(timestep=global_step, timesteps=total_steps)


def run_baseline_eval(
    agent, evaluator, eval_policy_fn, out_dir, state, divergence_baseline=None
):
    agent.enable_training_mode(False)
    eval_results = evaluator.evaluate(eval_policy_fn, out_dir=out_dir)
    agent.enable_training_mode(True)
    mean_return = np.mean([r["return"] for r in eval_results]) if eval_results else 0.0
    state.all_eval_returns.append(mean_return)
    state.eval_count += 1
    if state.wandb_enabled:
        log_eval_summary(
            mean_return,
            eval_results,
            state.all_eval_returns,
            state.eval_count,
            state.global_step,
            r_rand=divergence_baseline,
            consecutive_below=state.consecutive_below,
        )
    return mean_return


def check_baseline_divergence(mean_return, divergence_baseline, state):
    state.consecutive_below, diverged = check_divergence(
        mean_return, divergence_baseline, state.consecutive_below
    )
    if not diverged:
        return False
    state.diverged = True
    if state.wandb_enabled:
        log_metrics(
            {"eval/diverged": 1, "eval/diverge_step": state.global_step},
            step=state.global_step,
        )
    return state.diverged


def write_baseline_checkpoint(
    agent, out_dir, method_name, global_step, seed, artifacts, complete=False
):
    ckpt_dir = out_dir / (
        "checkpoint_final" if complete else f"checkpoint_{global_step}"
    )
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    agent.save(str(ckpt_dir / "skrl_agent.pt"))
    artifacts.append(str(ckpt_dir / "skrl_agent.pt"))
    manifest = {"method": method_name, "global_step": global_step, "seed": seed}
    if complete:
        manifest["complete"] = True
    with open(ckpt_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)


def finalize_baseline_run(env, agent, out_dir, method_name, seed, state, fps):
    write_baseline_checkpoint(
        agent,
        out_dir,
        method_name,
        state.global_step,
        seed,
        state.artifacts,
        complete=True,
    )
    csv_path = out_dir / "eval_history.csv"
    if csv_path.exists():
        state.artifacts.append(str(csv_path))
    reward_mean_last10 = (
        float(np.mean(state.all_eval_returns[-10:])) if state.all_eval_returns else 0.0
    )
    env_name = "unknown"
    if (
        hasattr(env, "unwrapped")
        and hasattr(env.unwrapped, "spec")
        and env.unwrapped.spec
    ):
        env_name = env.unwrapped.spec.id
    return {
        "method": method_name,
        "env_name": env_name,
        "seed": seed,
        "reward_mean_last10": reward_mean_last10,
        "fps": fps,
        "artifacts": state.artifacts,
        "diverged": state.diverged,
        "global_step": state.global_step,
    }


def run_baseline_iteration(
    env,
    agent,
    evaluator,
    eval_policy_fn,
    out_dir,
    method_name,
    seed,
    divergence_baseline,
    eval_steps,
    save_steps,
    total_steps,
    observations,
    states,
    state,
):
    """Run one act→step→record step plus scheduled eval/checkpoint; return (obs, states, diverged)."""
    agent.pre_interaction(timestep=state.global_step, timesteps=total_steps)
    actions = query_agent_actions(
        agent, observations, states, state.global_step, total_steps
    )
    next_obs, reward, terminated, truncated, next_states = apply_env_step(env, actions)
    record_agent_transition(
        agent,
        observations,
        states,
        actions,
        next_obs,
        reward,
        terminated,
        truncated,
        next_states,
        state.global_step,
        total_steps,
    )
    state.global_step += 1
    if terminated or truncated:
        observations, _ = env.reset()
        states = env.state() if hasattr(env, "state") else None
    else:
        observations = next_obs
        states = next_states
    if state.global_step % eval_steps == 0 or state.global_step >= total_steps:
        mean_return = run_baseline_eval(
            agent, evaluator, eval_policy_fn, out_dir, state, divergence_baseline
        )
        if check_baseline_divergence(mean_return, divergence_baseline, state):
            return observations, states, True
    if state.global_step % save_steps == 0 or state.global_step >= total_steps:
        write_baseline_checkpoint(
            agent, out_dir, method_name, state.global_step, seed, state.artifacts
        )
    return observations, states, False


def baseline_train_loop(
    env,
    agent,
    *,
    eval_env,
    out_dir,
    seed,
    total_steps,
    eval_steps,
    save_steps,
    method_name,
    eval_policy_fn,
    divergence_baseline=1.0,
    cfg_dict=None,
):
    """Run the skrl agent through a standard act→step→record→post_interaction loop.

    Returns the standard result dict (method, reward_mean_last10, artifacts, diverged).
    """
    evaluator, state = init_baseline_run(
        env, eval_env, out_dir, method_name, seed, total_steps, cfg_dict
    )
    agent.init()
    eval_policy_fn = resolve_eval_policy(agent, eval_policy_fn)
    agent.enable_training_mode(True)
    observations, _ = env.reset()
    states = env.state() if hasattr(env, "state") else None
    try:
        while state.global_step < total_steps:
            observations, states, diverged = run_baseline_iteration(
                env,
                agent,
                evaluator,
                eval_policy_fn,
                out_dir,
                method_name,
                seed,
                divergence_baseline,
                eval_steps,
                save_steps,
                total_steps,
                observations,
                states,
                state,
            )
            if diverged:
                break
        elapsed = time.time() - state.t_start
        fps = state.global_step / max(elapsed, 1e-6)
        if state.wandb_enabled:
            log_metrics(
                {"timing/fps": fps, "timing/elapsed_s": elapsed}, step=state.global_step
            )
    finally:
        if state.wandb_enabled:
            finish_run(exit_code=1 if state.diverged else 0)
    return finalize_baseline_run(env, agent, out_dir, method_name, seed, state, fps)


def build_ppo_skrl_config(cfg):
    """Build the skrl PPO_CFG from our PPOConfig (provenance in PROVENANCE_TABLE)."""
    return PPO_CFG_cls(
        rollouts=cfg.rollouts,
        learning_epochs=cfg.learning_epochs,
        mini_batches=cfg.mini_batches,
        discount_factor=cfg.discount,
        gae_lambda=cfg.gae_lambda,
        learning_rate=cfg.lr,
        learning_rate_scheduler=None,
        learning_rate_scheduler_kwargs={},
        observation_preprocessor=None,
        observation_preprocessor_kwargs={},
        state_preprocessor=None,
        state_preprocessor_kwargs={},
        value_preprocessor=None,
        value_preprocessor_kwargs={},
        grad_norm_clip=cfg.grad_norm_clip,
        ratio_clip=cfg.ratio_clip,
        value_clip=0.2,
        entropy_loss_scale=0.0,
        value_loss_scale=cfg.value_loss_scale,
        kl_threshold=cfg.kl_threshold,
        random_timesteps=0,
        learning_starts=0,
        time_limit_bootstrap=True,
        mixed_precision=False,
    )


def build_ppo_agent(policy, value, memory, skrl_env, ppo_cfg, device):
    return PPO_cls(
        models={"policy": policy, "value": value},
        memory=memory,
        observation_space=skrl_env.observation_space,
        state_space=skrl_env.state_space,
        action_space=skrl_env.action_space,
        device=device,
        cfg=ppo_cfg,
    )


def launch_baseline_loop(
    skrl_env,
    agent,
    wrapped_env,
    out_dir,
    seed,
    total_steps,
    eval_steps,
    save_steps,
    method_name,
    eval_policy,
    r_rand,
    cfg,
):
    return baseline_train_loop(
        skrl_env,
        agent,
        eval_env=wrapped_env,
        out_dir=out_dir,
        seed=seed,
        total_steps=total_steps,
        eval_steps=eval_steps,
        save_steps=save_steps,
        method_name=method_name,
        eval_policy_fn=eval_policy,
        divergence_baseline=r_rand,
        cfg_dict=dataclasses.asdict(cfg),
    )


def ppo_train_run(
    env: Any,
    out_dir: str | Path,
    cfg: PPOConfig | None = None,
    *,
    seed: int = 0,
    total_steps: int = 2000,
    eval_steps: int = 1000,
    save_steps: int = 1000,
    n_obs_steps: int = 16,
) -> dict[str, Any]:
    """Run a single PPO training run using skrl 2.1.0.

    Returns the standard result dict (method, reward_mean_last10, artifacts, diverged).
    """
    import_skrl_dependencies()

    cfg = cfg or PPOConfig()
    out_dir = Path(out_dir)
    device = str(resolve_device())
    wrapped_env, skrl_env, obs_dim, action_dim = seed_and_wrap_env(env, seed, n_obs_steps)
    policy = maybe_compile(PPOPolicy(obs_dim, action_dim, cfg.hidden))
    value = maybe_compile(PPOValue(obs_dim, cfg.hidden))
    memory = RandomMemory_cls(memory_size=cfg.rollouts, num_envs=1, device=device)
    ppo_cfg = build_ppo_skrl_config(cfg)
    agent = build_ppo_agent(policy, value, memory, skrl_env, ppo_cfg, device)
    eval_policy = mean_action_eval_policy(agent)
    r_rand = compute_random_baseline(env, seed, n_obs_steps)
    return launch_baseline_loop(
        skrl_env,
        agent,
        wrapped_env,
        out_dir,
        seed,
        total_steps,
        eval_steps,
        save_steps,
        "ppo",
        eval_policy,
        r_rand,
        cfg,
    )


def build_sac_skrl_config(cfg):
    """Build the skrl SAC_CFG from our SACConfig (provenance in PROVENANCE_TABLE)."""
    return SAC_CFG_cls(
        gradient_steps=cfg.gradient_steps,
        batch_size=cfg.batch_size,
        discount_factor=cfg.discount,
        polyak=cfg.polyak,
        learning_rate=cfg.lr,
        learning_rate_scheduler=None,
        learning_rate_scheduler_kwargs={},
        observation_preprocessor=None,
        observation_preprocessor_kwargs={},
        state_preprocessor=None,
        state_preprocessor_kwargs={},
        random_timesteps=0,
        learning_starts=cfg.initial_random_steps,
        grad_norm_clip=0,
        learn_entropy=True,
        initial_entropy_value=0.2,
        target_entropy=None,
        mixed_precision=False,
    )


def build_sac_agent(
    policy,
    critic_1,
    critic_2,
    target_critic_1,
    target_critic_2,
    memory,
    skrl_env,
    sac_cfg,
    device,
):
    return SAC_cls(
        models={
            "policy": policy,
            "critic_1": critic_1,
            "critic_2": critic_2,
            "target_critic_1": target_critic_1,
            "target_critic_2": target_critic_2,
        },
        memory=memory,
        observation_space=skrl_env.observation_space,
        state_space=skrl_env.state_space,
        action_space=skrl_env.action_space,
        device=device,
        cfg=sac_cfg,
    )


def sac_train_run(
    env: Any,
    out_dir: str | Path,
    cfg: SACConfig | None = None,
    *,
    seed: int = 0,
    total_steps: int = 2000,
    eval_steps: int = 1000,
    save_steps: int = 1000,
    n_obs_steps: int = 16,
) -> dict[str, Any]:
    """Run a single SAC training run using skrl 2.1.0.

    Returns the standard result dict (method, reward_mean_last10, artifacts, diverged).
    """
    import_skrl_dependencies()

    cfg = cfg or SACConfig()
    out_dir = Path(out_dir)
    device = str(resolve_device())
    wrapped_env, skrl_env, obs_dim, action_dim = seed_and_wrap_env(env, seed, n_obs_steps)
    policy = maybe_compile(SACPolicy(obs_dim, action_dim, cfg.hidden))
    critic_1 = maybe_compile(SACCritic(obs_dim, action_dim, cfg.hidden))
    critic_2 = maybe_compile(SACCritic(obs_dim, action_dim, cfg.hidden))
    target_critic_1 = maybe_compile(SACCritic(obs_dim, action_dim, cfg.hidden))
    target_critic_2 = maybe_compile(SACCritic(obs_dim, action_dim, cfg.hidden))
    memory = RandomMemory_cls(memory_size=cfg.memory_size, num_envs=1, device=device)
    sac_cfg = build_sac_skrl_config(cfg)
    agent = build_sac_agent(
        policy,
        critic_1,
        critic_2,
        target_critic_1,
        target_critic_2,
        memory,
        skrl_env,
        sac_cfg,
        device,
    )
    eval_policy = mean_action_eval_policy(agent)
    r_rand = compute_random_baseline(env, seed, n_obs_steps)
    return launch_baseline_loop(
        skrl_env,
        agent,
        wrapped_env,
        out_dir,
        seed,
        total_steps,
        eval_steps,
        save_steps,
        "sac",
        eval_policy,
        r_rand,
        cfg,
    )

In [ ]:
%%writefile src/sjfpo/checkpointing.py
"""Policy checkpoint IO (HuggingFace style): save_pretrained/from_pretrained.

``save_policy_checkpoint`` writes ``config.json`` + ``model.safetensors``
(unwrapped eager keys, so artifacts are compile-mode independent) plus
``optimizer.pt`` and ``counters.json``; ``load_policy_checkpoint`` rebuilds
everything from the directory.
"""

from __future__ import annotations

import contextlib
import json
import os
from pathlib import Path

import torch
from huggingface_hub.constants import CONFIG_NAME, SAFETENSORS_SINGLE_FILE

OPTIMIZER_FILE = "optimizer.pt"
COUNTERS_FILE = "counters.json"


class ResumeError(RuntimeError):
    """Unresumable checkpoint (missing/corrupt files)."""


@contextlib.contextmanager
def unwrapped_policy(policy):
    """Temporarily expose a policy's eager submodules for checkpoint IO.

    Swap torch.compile wrappers for their inner modules (and swap back
    afterwards); params are shared, so training continues untouched and the
    artifact holds plain eager keys whatever SJFPO_COMPILE was.
    """
    swapped = []
    for name in ("actor", "old_actor", "critic"):
        module = getattr(policy, name, None)
        if getattr(module, "_orig_mod", None) is not None:
            setattr(policy, name, module._orig_mod)
            swapped.append((name, module))
    try:
        yield policy
    finally:
        for name, module in swapped:
            setattr(policy, name, module)


def save_policy_checkpoint(
    dir_path: str | os.PathLike,
    *,
    policy,
    optimizer: torch.optim.Optimizer,
    global_step: int,
) -> None:
    """Save through LeRobot's HubMixin (+ optimizer.pt, counters.json).

    ``model.safetensors`` holds unwrapped eager keys, so artifacts are
    compile-mode independent. ``config.json`` carries shapes + hypers.
    """
    from sjfpo.modeling_sjfpo import SJFPOPolicy  # lazy: keeps this module import-light

    if not isinstance(policy, SJFPOPolicy):
        raise ValueError(
            f"save_policy_checkpoint needs an SJFPOPolicy, got {type(policy).__name__}"
        )
    d = Path(dir_path)
    d.mkdir(parents=True, exist_ok=True)
    with unwrapped_policy(policy):
        policy.save_pretrained(d)
    torch.save(optimizer.state_dict(), d / OPTIMIZER_FILE)
    (d / COUNTERS_FILE).write_text(
        json.dumps(
            {"global_step": int(global_step), "step_count": int(policy.step_count)},
            indent=2,
        ),
        encoding="utf-8",
    )


def load_policy_checkpoint(
    dir_path: str | os.PathLike,
    *,
    device: str | torch.device | None = None,
):
    """Load a checkpoint from :func:`save_policy_checkpoint`.

        Rebuilds components from ``config.json`` (no caller dims); policy left in
    eval mode. Raises ResumeError on missing/unreadable files.
    """
    from sjfpo.modeling_sjfpo import SJFPOPolicy  # lazy: keeps this module import-light

    d = Path(dir_path)
    missing = [
        f
        for f in (CONFIG_NAME, SAFETENSORS_SINGLE_FILE, OPTIMIZER_FILE, COUNTERS_FILE)
        if not (d / f).is_file()
    ]
    if missing:
        raise ResumeError(f"incomplete policy checkpoint in {d}: missing {missing}")
    policy = SJFPOPolicy.from_pretrained(d)
    if device is not None:
        policy.to(device)
    optimizer = policy.config.get_optimizer_preset().build(
        policy.get_optim_params()["params"]
    )
    try:
        opt_state = torch.load(
            d / OPTIMIZER_FILE, map_location="cpu", weights_only=True
        )
    except Exception as e:
        raise ResumeError(f"failed to unpickle {d / OPTIMIZER_FILE}: {e}") from e
    optimizer.load_state_dict(opt_state)
    try:
        counters = json.loads((d / COUNTERS_FILE).read_text(encoding="utf-8"))
        policy.step_count = int(counters["step_count"])
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as e:
        raise ResumeError(f"corrupt counters in {d}: {e}") from e
    return (
        policy,
        optimizer,
        {"global_step": int(counters["global_step"]), "step_count": policy.step_count},
    )

In [ ]:
%%writefile src/sjfpo/collector.py
"""Chunked open-loop rollout collector.

The policy is queried ONCE per chunk; the first ``n_action_steps`` proposed
actions execute OPEN-LOOP (1 <= n <= H). Transitions are plain tensor dicts
with LeRobot-style keys; ``action`` is the FULL proposed chunk in real scaled
space ([H, action_dim], one row per predicted step; DCT happens downstream
in ``forward``). V(s) bootstraps from the stored next window, so no
next-action proposal is collected.

Boundary semantics: terminated stops at once (no bootstrap); truncated stops
bootstrap-eligible; budget-exhausted tails are marked truncated so GAE never
recurses past the collected stream. Actions are clipped to bounds before
execution and every clipped element is counted. On env error the
``on_error_flush`` hook runs first, then the original exception re-raises.
"""

from __future__ import annotations

from collections import deque
from collections.abc import Callable
from typing import TypedDict

import numpy as np
import torch

from lerobot.processor.converters import to_tensor
from lerobot.utils.constants import DONE, OBS_STATE, REWARD, TRUNCATED
from sjfpo.configuration_sjfpo import SJFPOConfig

_DEFAULTS = SJFPOConfig()  # bare policy config: single source for defaults

__all__ = ["Transition", "collect_chunks", "collect_chunks_vec"]


Transition = TypedDict(
    "Transition",
    {
        "observation.state": torch.Tensor,  # [T, obs] state window at replan time
        "action": torch.Tensor,  # [H, action_dim] real SCALED FULL proposed chunk
        "next.observation.state": torch.Tensor,  # [T, obs] window after execution
        "next.reward": torch.Tensor,  # [n_executed] per executed step
        "next.done": torch.Tensor,  # bool scalar, terminated only
        "next.truncated": torch.Tensor,  # bool scalar, bootstrap-eligible
        "executed_n_action_steps": torch.Tensor,  # int64 scalar, actually executed steps
        "clip_count": torch.Tensor,  # int64 scalar, clipped proposal elements
    },
)


def as_float_tensor(x: object) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        return x.detach().to(torch.float32)
    return to_tensor(np.asarray(x, dtype=np.float32))


def scale_and_clip_chunk(
    proposed: torch.Tensor, low: torch.Tensor, high: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor, int]:
    if bool(((high - low) <= 0).any()):
        raise ValueError("action bounds must satisfy high > low in every dim")
    clip_count = int(((proposed < low) | (proposed > high)).sum())
    executed = torch.clamp(proposed, min=low, max=high)
    scaled = 2.0 * (executed - low) / (high - low) - 1.0
    return executed, scaled, clip_count


def build_warmup_window(
    first_obs: torch.Tensor, n_obs_steps: int
) -> deque[torch.Tensor]:
    return deque((first_obs.clone() for _ in range(n_obs_steps)), maxlen=n_obs_steps)


def reset_chunk_window(env, n_obs_steps: int) -> deque[torch.Tensor]:
    obs, _ = env.reset()
    new_obs = as_float_tensor(obs)
    return deque((new_obs.clone() for _ in range(n_obs_steps)), maxlen=n_obs_steps)


def execute_open_loop_steps(
    env,
    executed: torch.Tensor,
    window: deque[torch.Tensor],
    executed_total: int,
    total_steps: int,
    n_action_steps: int,
) -> tuple[list[float], str]:
    rewards: list[float] = []
    done_kind = "continue"
    for i in range(n_action_steps):
        if executed_total >= total_steps:
            break
        obs, reward, terminated, truncated, info = env.step(executed[i].numpy())
        executed_total += 1
        rewards.append(float(reward))
        window.append(as_float_tensor(obs))
        if terminated:  # checked first: termination wins, never bootstrap
            done_kind = "terminated"
            break
        if truncated:
            done_kind = "truncated"
            break
    return rewards, done_kind


def finalize_transition(
    state: torch.Tensor,
    next_state: torch.Tensor,
    scaled: torch.Tensor,
    rewards: list[float],
    done_kind: str,
    clip_count: int,
    chunk_size: int,
    n_action_steps: int,
) -> Transition:
    if scaled.shape[0] != chunk_size:
        raise ValueError(
            f"proposed chunk has {scaled.shape[0]} rows, "
            f"expected chunk_size={chunk_size}"
        )
    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    n_executed = len(rewards)
    return {
        OBS_STATE: state,
        "action": scaled,
        "next.observation.state": next_state,
        REWARD: to_tensor(rewards, dtype=torch.float32),
        DONE: to_tensor(done_kind == "terminated", dtype=torch.bool),
        TRUNCATED: to_tensor(done_kind == "truncated", dtype=torch.bool),
        "executed_n_action_steps": to_tensor(n_executed, dtype=torch.int64),
        "clip_count": to_tensor(clip_count, dtype=torch.int64),
    }


def propose_and_execute_chunk(
    policy: Callable[[torch.Tensor], torch.Tensor],
    window: deque[torch.Tensor],
    low: torch.Tensor,
    high: torch.Tensor,
    env,
    executed_total: int,
    total_steps: int,
    chunk_size: int,
    n_action_steps: int,
) -> tuple[Transition, str]:
    state = torch.stack(tuple(window), dim=0)
    proposed = as_float_tensor(policy(state))
    executed, scaled, clip_count = scale_and_clip_chunk(proposed, low, high)
    rewards, done_kind = execute_open_loop_steps(
        env, executed, window, executed_total, total_steps, n_action_steps
    )
    next_state = torch.stack(tuple(window), dim=0)
    record = finalize_transition(
        state,
        next_state,
        scaled,
        rewards,
        done_kind,
        clip_count,
        chunk_size,
        n_action_steps,
    )
    return record, done_kind


def flush_on_collection_error(
    on_error_flush: Callable[[], None] | None,
) -> None:
    if on_error_flush is not None:
        try:
            on_error_flush()
        except Exception:
            pass


def collect_chunks_vec(
    envs,
    policy: Callable[[torch.Tensor], torch.Tensor],
    total_steps: int,
    action_low: float | np.ndarray,
    action_high: float | np.ndarray,
    *,
    n_obs_steps: int = _DEFAULTS.n_obs_steps,
    n_action_steps: int,  # required: no config default; training passes chunk_size
    seed: int | None = None,
    on_error_flush: Callable[[], None] | None = None,
) -> list[Transition]:
    """Vector twin of :func:`collect_chunks` (same records, same chaining).

    ``envs`` is a ``gym.vector.VectorEnv`` (NextStep autoreset): each env runs
    its own chunk loop (own window, prev-chain, budget), but proposals are one
    batched policy call and stepping is one vector call. Per-env budget is
    ``ceil(total_steps / num_envs)``, so the merged batch matches the
    single-env size while collection scales ~linearly with env count.
    ``policy`` takes ``[M, T, obs]`` and returns ``[M, H, action_dim]``.
    """
    n = int(envs.num_envs)
    target = -(-total_steps // n)  # ceil: per-env executed-step budget
    low = as_float_tensor(action_low)
    high = as_float_tensor(action_high)
    chunk_size = _DEFAULTS.action_horizon
    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    if seed is None:
        seed = int(torch.randint(0, 2**31, (1,)).item())
    act_dim = int(np.asarray(envs.single_action_space.shape).prod())
    try:
        obs, _ = envs.reset(seed=[seed + i for i in range(n)])
        windows = [
            build_warmup_window(as_float_tensor(o), n_obs_steps) for o in obs
        ]
        cur: list = [None] * n  # (executed, scaled, clip_count, replan_state)
        pos = [0] * n
        rewards: list[list[float]] = [[] for _ in range(n)]
        prev: list = [None] * n
        prev_done = ["continue"] * n
        executed_total = [0] * n
        awaiting = [False] * n  # done last step: rebuild window from reset obs
        records: list[list[Transition]] = [[] for _ in range(n)]
        while any(e < target for e in executed_total):
            need = [
                i
                for i in range(n)
                if cur[i] is None
                and not awaiting[i]
                and executed_total[i] < target
            ]
            if need:
                states = torch.stack(
                    [torch.stack(tuple(windows[i]), dim=0) for i in need]
                )
                proposed = as_float_tensor(policy(states))
                for k, i in enumerate(need):
                    executed, scaled, clip_count = scale_and_clip_chunk(
                        proposed[k], low, high
                    )
                    cur[i] = (executed, scaled, clip_count, states[k])
                    pos[i] = 0
                    rewards[i] = []
            actions = np.zeros((n, act_dim), dtype=np.float32)
            for i in range(n):
                if cur[i] is not None:
                    actions[i] = cur[i][0][pos[i]].numpy()
            obs, rew, term, trunc, _ = envs.step(actions)
            done = np.asarray(term, dtype=bool) | np.asarray(trunc, dtype=bool)
            for i in range(n):
                if awaiting[i]:
                    # Gap step after done: this obs is the fresh reset obs.
                    windows[i] = build_warmup_window(
                        as_float_tensor(obs[i]), n_obs_steps
                    )
                    awaiting[i] = False
                    continue
                if cur[i] is None:
                    continue  # over-budget env idling on dummy actions
                rewards[i].append(float(rew[i]))
                windows[i].append(as_float_tensor(obs[i]))
                pos[i] += 1
                if done[i] or pos[i] >= n_action_steps:
                    executed, scaled, clip_count, replan = cur[i]
                    n_exec = pos[i]
                    if bool(term[i]):
                        done_kind = "terminated"
                    elif bool(trunc[i]):
                        done_kind = "truncated"
                    else:
                        done_kind = "continue"
                    next_state = torch.stack(tuple(windows[i]), dim=0)
                    record = finalize_transition(
                        replan,
                        next_state,
                        scaled,
                        rewards[i],
                        done_kind,
                        clip_count,
                        chunk_size,
                        n_action_steps,
                    )
                    executed_total[i] += n_exec
                    records[i].append(record)
                    prev[i], prev_done[i] = record, done_kind
                    cur[i] = None
                    if done[i]:
                        awaiting[i] = True
        tails = [i for i in range(n) if prev[i] is not None and prev_done[i] == "continue"]
        for i in tails:
            # Budget exhausted mid-episode: bootstrap the cut, but never
            # recurse into another env's GAE.
            prev[i][TRUNCATED] = torch.tensor(True)
        return [record for trajectory in records for record in trajectory]
    except Exception:
        flush_on_collection_error(on_error_flush)
        raise


def collect_chunks(
    env,
    policy: Callable[[torch.Tensor], torch.Tensor],
    total_steps: int,
    action_low: float | np.ndarray,
    action_high: float | np.ndarray,
    *,
    n_obs_steps: int = _DEFAULTS.n_obs_steps,
    n_action_steps: int,  # required: no config default; training passes chunk_size
    on_error_flush: Callable[[], None] | None = None,
) -> list[Transition]:
    low = as_float_tensor(action_low)
    high = as_float_tensor(action_high)
    chunk_size = _DEFAULTS.action_horizon
    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    try:
        obs, _ = env.reset()
        first_obs = as_float_tensor(obs)
        window = build_warmup_window(first_obs, n_obs_steps)
        records: list[Transition] = []
        executed_total = 0
        while executed_total < total_steps:
            record, done_kind = propose_and_execute_chunk(
                policy,
                window,
                low,
                high,
                env,
                executed_total,
                total_steps,
                chunk_size,
                n_action_steps,
            )
            executed_total += int(record["executed_n_action_steps"])
            records.append(record)
            if done_kind != "continue":
                window = reset_chunk_window(env, n_obs_steps)
        return records
    except Exception:
        flush_on_collection_error(on_error_flush)
        raise

In [ ]:
%%writefile src/sjfpo/config.py
"""Benchmark config registry: PPO/SAC hyperparameters + run matrix.

Dataclasses are frozen with tuple defaults; RUN_MATRIX is a tuple of frozen
RunSpec built over methods × envs × seeds. Mode switches read env vars once
at import. MuJoCo-only (flat Box).
"""

from __future__ import annotations

import os
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path

# Mode switches (plain module-level constants; read env at import time ONLY)

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
TINY_MATRIX = os.environ.get("SJFPO_TINY_MATRIX") == "1"

METHODS = ("sjfpo", "ppo", "sac")
ENVS = ("HalfCheetah-v5", "Walker2d-v5", "Hopper-v5")
PROTOTYPE_ENVS = (
    "InvertedPendulum-v5",
    "InvertedDoublePendulum-v5",
    "Reacher-v5",
    "Swimmer-v5",
    "Pusher-v5",
    "Ant-v5",
    "Humanoid-v5",
    "AdroitHandDoor-v1",
    "AdroitHandPen-v1",
    "AdroitHandHammer-v1",
)
KNOWN_ENVS = ENVS + PROTOTYPE_ENVS  # --env choices; matrix still uses ENVS only
ENV_REGISTRY_PATH = Path(__file__).resolve().parents[2] / "configs" / "envs.yaml"
SEEDS = (0, 1, 2, 3, 4)

TOTAL_STEPS = 1_000_000
EVAL_STEPS = 50_000
SAVE_STEPS = 25_000


# Method hyperparameters (Scope §Must-have, verbatim values)


@dataclass(frozen=True)
class PPOConfig:
    """skrl 2.1.0 PPO baseline keys."""

    rollouts: int = 2048
    learning_epochs: int = 10
    mini_batches: int = 8
    lr: float = 3e-4
    kl_threshold: float = 0.01  # KLAdaptiveLR
    grad_norm_clip: float = 1.0
    discount: float = 0.99
    gae_lambda: float = 0.95
    ratio_clip: float = 0.2
    value_loss_scale: float = 1.0
    hidden: tuple[int, ...] = (256, 256, 256)


@dataclass(frozen=True)
class SACConfig:
    """skrl 2.1.0 SAC baseline keys."""

    lr: float = 3e-4
    batch_size: int = 256
    memory_size: int = 300_000
    gradient_steps: int = 1
    polyak: float = 0.005
    discount: float = 0.99
    initial_random_steps: int = 5000
    hidden: tuple[int, ...] = (256, 256, 256)


PPO_CFG = PPOConfig()
SAC_CFG = SACConfig()


@dataclass(frozen=True)
class RunSpec:

    method: str
    env: str
    seed: int
    total_steps: int
    eval_steps: int
    save_steps: int


def build_run_matrix() -> tuple[RunSpec, ...]:
    """Legacy MuJoCo matrix (45 runs: 3 methods × 3 envs × 5 seeds)."""
    specs: list[RunSpec] = []
    for method in METHODS:
        for env in ENVS:
            for seed in SEEDS:
                specs.append(
                    RunSpec(
                        method=method,
                        env=env,
                        seed=seed,
                        total_steps=TOTAL_STEPS,
                        eval_steps=EVAL_STEPS,
                        save_steps=SAVE_STEPS,
                    )
                )
    return tuple(specs)


RUN_MATRIX = build_run_matrix()  # 45 runs


if TINY_MATRIX:
    tiny: list[RunSpec] = []
    for method in METHODS:
        tiny.append(
            RunSpec(
                method=method,
                env="Hopper-v5",
                seed=0,
                total_steps=20_000,
                eval_steps=10_000,
                save_steps=10_000,
            )
        )
    RUN_MATRIX = tuple(tiny)  # 3 runs


def validate_run_matrix(specs: Iterable[RunSpec]) -> None:
    """Raise AssertionError on any duplicate (method, env, seed)."""
    seen: set[tuple[str, str, int]] = set()
    for spec in specs:
        key = (spec.method, spec.env, spec.seed)
        assert key not in seen, f"duplicate (method, env, seed) in run matrix: {key}"
        seen.add(key)


def parse_version_tuple(version: str) -> tuple[int, ...]:
    """Version string -> (major, minor, patch, ...) for comparison."""
    parts: list[int] = []
    for part in version.split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def assert_runtime(allow_fp16: bool = False) -> None:
    """Fail fast on dependency pins and unsupported precision flags.

    bf16 *autocast* (fp32 storage, bf16 matmuls) is allowed and needs no
    flag; full half/bfloat STORAGE raises.
    """
    import importlib.metadata

    gym_version = importlib.metadata.version("gymnasium")
    if parse_version_tuple(gym_version)[:2] < (1, 0):
        raise RuntimeError(
            f"gymnasium {gym_version} has major.minor < 1.0; "
            "this project requires gymnasium[mujoco]>=1.0,<1.3"
        )

    skrl_version = importlib.metadata.version("skrl")
    if skrl_version != "2.1.0":
        raise RuntimeError(f"skrl {skrl_version} != 2.1.0; exact pin required")

    if allow_fp16:
        raise RuntimeError(
            "fp16/bf16 storage requested but unsupported: flow-matching "
            "Euler rollouts and CFM losses require fp32 storage; use "
            "SJFPO_DTYPE=bf16 for bf16 autocast (fp32 params, bf16 matmuls)"
        )


validate_run_matrix(RUN_MATRIX)


def lookup_env_dims(env_id: str) -> tuple[int, int]:
    """(obs_dim, action_dim) from configs/envs.yaml; KeyError on unknown ids."""
    try:
        import yaml
    except ImportError:
        raise RuntimeError("pyyaml required for --env lookup (pip install pyyaml)")
    with open(ENV_REGISTRY_PATH) as f:
        registry = yaml.safe_load(f)
    try:
        entry = registry[env_id]
    except KeyError:
        raise KeyError(f"unknown env {env_id!r}; add it to {ENV_REGISTRY_PATH}")
    return int(entry["obs_dim"]), int(entry["action_dim"])

In [ ]:
%%writefile src/sjfpo/configuration_sjfpo.py
"""Single home for every SJFPO hyperparameter (LeRobot PreTrainedConfig).

Runs serialize/resume through HubMixin + draccus (``type = "sjfpo"``).
Legacy names (``config.py::SJFPOConfig``): horizon/chunk_size (now action_horizon),
optimizer_lr/lr, optimizer_weight_decay/wd,
grad_clip_norm/grad_clip. ``n_action_steps`` is the intended inference prefix
(``round(executed_vs_predicted_ratio * action_horizon)``); training executes the
full action_horizon open-loop.
"""

from __future__ import annotations

from dataclasses import dataclass

from lerobot.configs import PreTrainedConfig
from lerobot.optim import AdamWConfig


@PreTrainedConfig.register_subclass("sjfpo")
@dataclass
class SJFPOConfig(PreTrainedConfig):
    """SJFPO hyperparameters (see field names; all self-explanatory)."""

    n_obs_steps: int = 16
    action_horizon: int = 16
    n_action_steps: int = 8
    hidden_dim: int = 128

    num_mc_samples: int = 16
    num_euler_steps: int = 16
    num_inference_steps: int = 16
    chunks_per_update: int = 128
    num_epochs: int = 16
    num_minibatches: int = 4
    num_envs: int = 8  # parallel rollout environments; 1 uses single-env collection

    gamma: float = 0.99
    gae_lambda: float = 0.95
    epsilon_clip: float = 0.05
    use_per_bin_clip: bool = False  # Phase 1: two-group epsilon over DCT bins
    epsilon_clip_lowfreq: float = 0.02  # tight: low bins carry task intent
    epsilon_clip_highfreq: float = 0.10  # loose: high bins are detail/exploration
    cfm_clamp: float = 20.0
    log_ratio_cap: float = 10.0
    advantage_clamp: float = 100.0
    use_clipped_value_loss: bool = False
    min_timestep: float = 0.005
    max_timestep: float = 0.995

    dit_depth: int = 8
    dit_heads: int = 4
    dit_mlp_ratio: float = 4.0
    dit_time_dim: int = 256
    timestep_scale: float = 1000.0  # Flow time t in [0,1] is scaled by this before
    # the DDPM sinusoidal ladder (designed for integer steps 0-1000); without it
    # every channel sits at angle <= 1 and t is nearly unresolvable.

    vf_coef: float = 1.0
    grad_clip_norm: float = 1.0

    top_p_spectral_tokens: float = 0.3
    executed_vs_predicted_ratio: float = 0.5

    optimizer_lr: float = 1e-4
    optimizer_weight_decay: float = 1e-4
    optimizer_8bit: bool = False

    def __post_init__(self):
        super().__post_init__()
        if self.num_euler_steps < 1:
            raise ValueError(f"num_euler_steps={self.num_euler_steps} must be >= 1")
        if self.num_inference_steps < 1:
            raise ValueError(f"num_inference_steps={self.num_inference_steps} must be >= 1")
        if not 0 < self.n_action_steps <= self.action_horizon:
            raise ValueError(
                f"n_action_steps={self.n_action_steps} must satisfy "
                f"0 < n <= action_horizon={self.action_horizon}"
            )
        r = self.executed_vs_predicted_ratio
        if not 0.0 < r <= 1.0:
            raise ValueError(f"executed_vs_predicted_ratio={r} must satisfy 0 < r <= 1")
        p = self.top_p_spectral_tokens
        if not 0.0 < p <= 1.0:
            raise ValueError(f"top_p_spectral_tokens={p} must satisfy 0 < p <= 1")
        if not 0.0 < self.epsilon_clip < 1.0:
            raise ValueError(
                f"require 0 < epsilon_clip < 1, got ({self.epsilon_clip})"
            )
        if not 0.0 < self.epsilon_clip_lowfreq <= self.epsilon_clip_highfreq < 1.0:
            raise ValueError(
                "require 0 < epsilon_clip_lowfreq <= epsilon_clip_highfreq < 1, got "
                f"({self.epsilon_clip_lowfreq}, {self.epsilon_clip_highfreq})"
            )
        if self.cfm_clamp <= 0:
            raise ValueError(f"cfm_clamp={self.cfm_clamp} must be positive")
        if self.log_ratio_cap <= 0:
            raise ValueError(f"log_ratio_cap={self.log_ratio_cap} must be positive")
        if self.advantage_clamp <= 0:
            raise ValueError(
                f"advantage_clamp={self.advantage_clamp} must be positive"
            )
        if not 0.0 <= self.min_timestep < self.max_timestep <= 1.0:
            raise ValueError(
                "require 0 <= min_timestep < max_timestep <= 1, got "
                f"({self.min_timestep}, {self.max_timestep})"
            )
        if self.timestep_scale <= 0:
            raise ValueError(f"timestep_scale={self.timestep_scale} must be positive")
        if self.optimizer_lr <= 0:
            raise ValueError(
                f"require optimizer_lr > 0, got ({self.optimizer_lr})"
            )
        if self.num_envs < 1:
            raise ValueError(f"num_envs={self.num_envs} must be >= 1")

    def validate_features(self) -> None:
        """Require observation.state + action features (base class skips this)."""
        if self.robot_state_feature is None:
            raise ValueError("SJFPO requires 'observation.state' in input_features.")
        if self.action_feature is None:
            raise ValueError("SJFPO requires 'action' in output_features.")

    def get_optimizer_preset(self) -> AdamWConfig:
        return AdamWConfig(
            lr=self.optimizer_lr,
            weight_decay=self.optimizer_weight_decay,
            grad_clip_norm=self.grad_clip_norm,
        )

    def get_scheduler_preset(self):
        """Return a LRSchedulerConfig from lerobot.optim, or None."""
        return None

    @property
    def observation_delta_indices(self) -> list[int] | None:
        """Stacked past frames ending at the current step."""
        return list(range(1 - self.n_obs_steps, 1))

    @property
    def action_delta_indices(self) -> list[int]:
        """Action-chunk offsets the dataset loader returns."""
        return list(range(self.action_horizon))

    @property
    def reward_delta_indices(self) -> None:
        """None: rollouts supply executed-step rewards directly."""
        return None

In [ ]:
%%writefile src/sjfpo/dct_utils.py
"""Orthonormal DCT-II / DCT-III utilities (pure torch matmuls, no scipy).

M[k, n] = sqrt(2/N)·c_k·cos(π(2n+1)k/2N), c_0 = 1/√2. Orthonormality (MᵀM = I)
preserves Euclidean measure, so Gaussians and log-densities survive the
change of variables (log-det = 0). ``dct(x) = x @ Mᵀ``, ``idct(y) = y @ M``.
"""

from __future__ import annotations

import math

import einx
import torch

__all__ = [
    "dct",
    "idct",
    "discounted_chunk_reward",
]

BASIS_CACHE: dict[int, torch.Tensor] = {}


def dct_basis_matrix(n: int) -> torch.Tensor:
    """[n, n] orthonormal DCT-II basis M (MᵀM = I)."""
    m = BASIS_CACHE.get(n)
    if m is None:
        k = einx.id("n -> n 1", torch.arange(n, dtype=torch.float32))
        t = einx.id("n -> 1 n", torch.arange(n, dtype=torch.float32))
        m = math.sqrt(2.0 / n) * torch.cos(math.pi * (2.0 * t + 1.0) * k / (2.0 * n))
        m[0].mul_(torch.rsqrt(torch.tensor(2.0)))
        BASIS_CACHE[n] = m
    return m


def apply_matrix_along_dim(
    x: torch.Tensor, dim: int, mat: torch.Tensor
) -> torch.Tensor:
    """Right-multiply the ``dim`` slice of ``x`` by ``mat`` (same shape out)."""
    dim = dim % x.dim()
    if dim != x.dim() - 1:
        x = x.movedim(dim, -1)
    out = x @ mat.to(x.device)
    if dim != out.dim() - 1:
        out = out.movedim(-1, dim)
    return out


def dct(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    return apply_matrix_along_dim(x, dim, dct_basis_matrix(x.shape[dim % x.dim()]).t())


def idct(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Inverse (DCT-III) along ``dim``: matmul with M (M⁻¹ = Mᵀ)."""
    return apply_matrix_along_dim(x, dim, dct_basis_matrix(x.shape[dim % x.dim()]))


def discounted_chunk_reward(rewards: torch.Tensor, gamma: float) -> torch.Tensor:
    n = rewards.shape[-1]
    weights = gamma ** torch.arange(n, dtype=rewards.dtype, device=rewards.device)
    return (rewards * weights).sum(dim=-1)

In [ ]:
%%writefile src/sjfpo/evaluator.py
"""SJFPO evaluator — deterministic rollouts, ``eval_history.csv``.

Columns ``[episode, return, length, success, jerk]``: ``success`` is the env's
``info["success"]`` when provided (else empty); ``jerk`` is mean squared
second-difference of the episode actions (else empty for < 3 steps).
"""

from __future__ import annotations

import csv
from collections.abc import Callable
from pathlib import Path
from typing import Any

import numpy as np


def write_eval_csv(results: list[dict[str, Any]], out_dir: str | Path) -> None:
    """Write ``eval_history.csv`` with columns ``[episode, return, length, success, jerk]``."""
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    csv_path = out_path / "eval_history.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["episode", "return", "length", "success", "jerk"]
        )
        writer.writeheader()
        writer.writerows(results)


def action_jerk(actions: np.ndarray) -> float | None:
    """Mean squared second-difference of actions [T, D] (None when T < 3).

    Scale-free smoothness probe: constant or constant-velocity action streams
    score exactly 0; contact jitter and bang-bang switching score high.
    """
    a = np.asarray(actions, dtype=np.float64)
    if a.ndim != 2 or a.shape[0] < 3:
        return None
    return float(np.square(a[2:] - 2.0 * a[1:-1] + a[:-2]).mean())


def extract_success(info: dict) -> float | None:
    """Env-reported task success (``info["success"]``), None when absent."""
    if not isinstance(info, dict) or info.get("success") is None:
        return None
    return float(bool(info["success"]))


def mean_chunk_jerk(actions: np.ndarray, is_pad: np.ndarray) -> float | None:
    """Mean ``action_jerk`` over chunks' executed prefixes.

    ``actions`` [N, H, D] real-space chunks, ``is_pad`` [N, H] tail mask.
    None when no chunk executed >= 3 steps.
    """
    a = np.asarray(actions)
    pad = np.asarray(is_pad, dtype=bool)
    jerks = []
    for chunk, mask in zip(a, pad):
        n_exec = int((~mask).sum())
        if n_exec >= 3:
            j = action_jerk(chunk[:n_exec])
            if j is not None:
                jerks.append(j)
    return float(np.mean(jerks)) if jerks else None


def render_episode_frames(
    env_id: str,
    policy: Callable[[np.ndarray], np.ndarray],
    seed: int = 0,
    max_steps: int = 1000,
) -> tuple[list[np.ndarray], float]:
    """One deterministic episode frames [T,H,W,C] uint8 via rgb_array + return."""
    import gymnasium as gym

    env = gym.make(env_id, render_mode="rgb_array")
    try:
        obs, _ = env.reset(seed=seed)
        if hasattr(policy, "reset"):
            policy.reset()
        frames: list[np.ndarray] = []
        ret = 0.0
        for _ in range(max_steps):
            action = policy(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            ret += float(reward)
            frames.append(np.asarray(env.render(), dtype=np.uint8))
            if terminated or truncated:
                break
        return frames, ret
    finally:
        env.close()


def check_divergence(
    mean_return: float, baseline: float, consecutive: int
) -> tuple[int, bool]:
    """Count sub-10% evals; diverged after 3 straight (never on baseline <= 0)."""
    if baseline > 0 and mean_return < 0.10 * baseline:
        consecutive += 1
    else:
        consecutive = 0
    return consecutive, consecutive >= 3


def compute_random_baseline(env: Any, seed: int, n_obs_steps: int) -> float:
    """Random-policy baseline return for divergence detection (1.0 if empty/zero)."""
    import gymnasium as gym

    from sjfpo.baselines import FrameStackWrapper

    fresh_env = (
        gym.make(env.unwrapped.spec.id)
        if hasattr(env, "unwrapped")
        and hasattr(env.unwrapped, "spec")
        and env.unwrapped.spec is not None
        else type(env.unwrapped)()
    )
    random_env = FrameStackWrapper(fresh_env, n_obs_steps=n_obs_steps)
    results = Evaluator(random_env, eval_seed=10000 + seed, max_episodes=12).evaluate(
        lambda obs: env.action_space.sample(), out_dir=None
    )
    r_rand = np.mean([r["return"] for r in results]) if results else 1.0
    random_env.close()
    return r_rand or 1.0


class Evaluator:
    """Deterministic evaluator for flat-observation (MuJoCo) envs."""

    def __init__(
        self,
        env: Any,
        eval_seed: int = 0,
        max_episodes: int = 12,
        max_steps: int | None = None,
    ):
        if env is not None and hasattr(env, "observation_space"):
            try:
                import gymnasium as gym
            except ImportError as e:
                raise ImportError(
                    "gymnasium is required for Evaluator obs-contract validation"
                ) from e
            if isinstance(getattr(env, "observation_space"), gym.spaces.Dict):
                raise AssertionError("Evaluator expects flat Box obs")
        self.env = env
        self.eval_seed = eval_seed
        self.max_episodes = max_episodes
        self.max_steps = max_steps

    def rollout_single_episode(
        self,
        policy: Callable[[np.ndarray], np.ndarray],
        seed: int,
        episode: int,
        step_budget: int | None,
    ) -> tuple[dict[str, Any], int]:
        """Roll one deterministic episode; return (result, steps taken)."""
        obs, _ = self.env.reset(seed=seed)
        if hasattr(policy, "reset"):
            policy.reset()
        ep_return = 0.0
        ep_length = 0
        steps_taken = 0
        done = False
        ep_actions: list[np.ndarray] = []
        ep_success: float | None = None
        while not done:
            action = policy(obs)
            obs, reward, terminated, truncated, info = self.env.step(action)
            ep_return += float(reward)
            ep_length += 1
            steps_taken += 1
            done = terminated or truncated
            ep_actions.append(np.asarray(action, dtype=np.float64).reshape(-1))
            success = extract_success(info)
            if success is not None:
                ep_success = success  # last report wins (terminal info)
            if step_budget is not None and steps_taken >= step_budget:
                break
        return {
            "episode": episode,
            "return": ep_return,
            "length": ep_length,
            "success": ep_success,
            "jerk": action_jerk(np.stack(ep_actions)) if ep_actions else None,
        }, steps_taken

    def evaluate(
        self,
        policy: Callable[[np.ndarray], np.ndarray],
        *,
        out_dir: str | Path | None = None,
    ) -> list[dict[str, Any]]:
        """Run deterministic rollouts, optionally writing ``eval_history.csv``."""
        rng = np.random.RandomState(self.eval_seed)
        results: list[dict[str, Any]] = []
        total_steps = 0
        for ep in range(self.max_episodes):
            if self.max_steps is not None and total_steps >= self.max_steps:
                break
            seed = int(rng.randint(0, 2**31))
            budget = None if self.max_steps is None else self.max_steps - total_steps
            result, taken = self.rollout_single_episode(policy, seed, ep, budget)
            results.append(result)
            total_steps += taken
        if out_dir is not None:
            write_eval_csv(results, out_dir)
        return results

In [ ]:
%%writefile src/sjfpo/fpo_core.py
"""FPO++ core math: CFM losses, per-sample ratios, ASPO trust region, Euler sampler (arXiv 2602.02481 Eq.3-13)."""
from __future__ import annotations

from collections.abc import Callable
from typing import Any

import einx
import math
import torch
from sjfpo.configuration_sjfpo import SJFPOConfig


# Single source for ``None``-fallback defaults.
_DEFAULTS = SJFPOConfig()


def sample_timesteps(
    num_samples: int,
    tau_range=None,
    device=None,
    mode="uniform",
    *,
    alpha=None,
    beta=None,
):
    """Flow times τ for the shared MC draw: [num_samples] in [lo, hi].

    ``mode``: "uniform" = U[lo, hi] (default); "normal" = N(mid, range/4)
    clamped, oversampling mid-times where the field curves most;
    "beta" = lerobot openpi Beta(alpha, beta) scaled to [lo, hi]
    (needs alpha=/beta= kwargs).
    """
    lo, hi = (
        (_DEFAULTS.min_timestep, _DEFAULTS.max_timestep)
        if tau_range is None
        else tau_range
    )
    if mode == "uniform":
        return torch.rand(num_samples, device=device) * (hi - lo) + lo
    if mode == "normal":
        mid = 0.5 * (lo + hi)
        std = 0.25 * (hi - lo)
        return (torch.randn(num_samples, device=device) * std + mid).clamp(lo, hi)
    if mode == "beta":
        if alpha is None or beta is None:
            raise ValueError("mode='beta' needs alpha= and beta= kwargs")
        from lerobot.policies.common.flow_matching import sample_time_beta

        return sample_time_beta(
            num_samples, device, alpha=alpha, beta=beta, scale=hi - lo, offset=lo
        )
    raise ValueError(f"mode={mode!r} unknown; expected 'uniform', 'normal' or 'beta'")


def sample_noise(shape, device=None, mode="normal"):
    """MC noise ε of shape [*shape].

    ``mode``: "normal" = N(0, I) (CFM-correct, v* assumes Gaussian endpoints);
    "uniform" = U[-√3, √3] (unit variance, same velocity scale; ablations).
    """
    if mode == "normal":
        return torch.randn(*shape, device=device)
    if mode == "uniform":
        s = math.sqrt(3.0)
        return (torch.rand(*shape, device=device) * 2.0 - 1.0) * s
    raise ValueError(f"mode={mode!r} unknown; expected 'normal' or 'uniform'")


def run_mc_simulation(a: torch.Tensor, eps: torch.Tensor, tau):
    """Broadcast (a, eps, tau) onto the shared [B*num_mc_samples] MC axis.

    tau: scalar, [num_mc_samples], or [B, num_mc_samples].
    Returns (tau_bn, eps_flat, a_rep, flow_shape, num_mc); raises ValueError
    on shape mismatches.
    """
    b, num_mc = eps.shape[0], eps.shape[1]
    flow_shape = tuple(a.shape[1:])
    if tuple(eps.shape) != (b, num_mc, *flow_shape):
        raise ValueError(f"eps shape {tuple(eps.shape)} != {(b, num_mc, *flow_shape)}")
    t = torch.as_tensor(tau, dtype=torch.float32)
    if t.dim() == 0:
        t = einx.id(" -> b m", t, b=b, m=num_mc)
    elif t.dim() == 1:
        if t.numel() != num_mc:
            raise ValueError(
                f"tau vector must have num_mc_samples={num_mc} entries, got {t.numel()}"
            )
        t = einx.id("m -> b m", t, b=b)
    elif tuple(t.shape) != (b, num_mc):
        raise ValueError(
            f"tau shape {tuple(t.shape)} must be scalar, [num_mc_samples] or [B, num_mc_samples]"
        )
    d = a.numel() // b
    a_flat = a.reshape(b, d)
    return (
        einx.id("b m -> (b m)", t),
        eps.reshape(b * num_mc, d),
        einx.id("b d -> (b m) d", a_flat, m=num_mc),
        flow_shape,
        num_mc,
    )


def compute_cfm_loss(
    v_hat: torch.Tensor,
    v_star: torch.Tensor,
    reduce_mode: str = "scalar",
    clamp: float | None = None,
) -> torch.Tensor:
    """CFM squared-velocity loss with selectable reduction (paper Eq.7-8).

    OFF ``sqrt`` convention (``rl_cfg.cfm_loss_reduction`` default): per-draw
    losses are ``sum / sqrt(D)`` over flow dims, which preserves variance
    across action dims instead of shrinking log-ratios like a mean. ``bin``
    applies the same scaling per frequency bin (diagnostic path only).
    Symmetric max-clamp (OFF ``cfm_loss_clamp``) applies to ``sample``/``bin``.
    """
    if v_hat.shape != v_star.shape:
        raise ValueError(
            f"v_hat shape {tuple(v_hat.shape)} != v_star shape {tuple(v_star.shape)}"
        )
    squared_errors = (v_hat - v_star).pow(2)
    if reduce_mode == "scalar":
        if clamp is not None:
            raise ValueError("clamp applies to reduce_mode='sample'/'bin' only")
        return squared_errors.mean()
    elif reduce_mode == "sample":
        flat = squared_errors.flatten(2)
        reduced = flat.sum(dim=-1) / math.sqrt(flat.shape[-1])
    elif reduce_mode == "bin":
        reduced = squared_errors.sum(dim=-1) / math.sqrt(squared_errors.shape[-1])
    else:
        raise ValueError(
            f"reduce_mode={reduce_mode!r} must be 'scalar', 'sample', or 'bin'"
        )

    clamp = _DEFAULTS.cfm_clamp if clamp is None else clamp
    return torch.clamp(reduced, max=clamp)


def compute_aspo_surrogate(
    rho: torch.Tensor,
    advantage: torch.Tensor,
    epsilon_clip: float | None = None,
) -> torch.Tensor:
    """ASPO piecewise trust region ψ(ρ, Â) (paper Eq.12).

    Single clipping parameter for both branches: Â > 0 PPO-clip,
    Â ≤ 0 SPO quadratic. Elementwise; scalars and tensors both supported.
    """
    epsilon_clip = _DEFAULTS.epsilon_clip if epsilon_clip is None else epsilon_clip
    pos = torch.min(
        rho * advantage,
        torch.clamp(rho, 1.0 - epsilon_clip, 1.0 + epsilon_clip) * advantage,
    )
    neg = rho * advantage - (advantage.abs() / (2.0 * epsilon_clip)) * (rho - 1.0) ** 2
    return torch.where(advantage > 0, pos, neg)


def compute_policy_ratio(
    l_old: torch.Tensor,
    l_new: torch.Tensor,
    cfm_clamp: float | None = None,
    cap: float | None = None,
) -> torch.Tensor:
    """Per-sample importance ratios (paper Eq.10 + OFF clamps).

    ρ_i = exp(ℓ_old,i − ℓ_new,i) with OFF's stabilizers: symmetric max-clamp
    on both losses (``cfm_loss_clamp``) and an STE cap on the log-ratio
    (forward clamped, gradient flows; ``cfm_diff_clamp_max``). OFF's separate
    negative-advantage max coincides with ``cfm_clamp`` here (single knob),
    so the symmetric clamp already covers it.
    """
    cfm_clamp = _DEFAULTS.cfm_clamp if cfm_clamp is None else cfm_clamp
    l_old_c = torch.clamp(l_old, max=cfm_clamp)
    l_new_c = torch.clamp(l_new, max=cfm_clamp)
    cap = _DEFAULTS.log_ratio_cap if cap is None else cap
    delta = l_old_c - l_new_c
    delta_st = delta + (torch.clamp(delta, max=cap) - delta).detach()
    return torch.exp(delta_st)


def compute_policy_loss(
    l_old: torch.Tensor,
    l_new: torch.Tensor,
    advantage: torch.Tensor,
    epsilon_clip: float | None = None,
    cfm_clamp: float | None = None,
    cap: float | None = None,
) -> torch.Tensor:
    """Scalar FPO++ policy objective: MEAN over i of ψ per chunk (Eq.13).

    Per-sample ratios enter the mean directly — never an exp-of-mean estimator.
    """
    rho = compute_policy_ratio(l_old, l_new, cfm_clamp=cfm_clamp, cap=cap)
    return compute_aspo_surrogate(rho, advantage, epsilon_clip).mean()


def sample_action(
    predict_velocity: Callable[[torch.Tensor, torch.Tensor, Any], torch.Tensor],
    shape,
    condition,
    num_inference_steps: int,
    eps: torch.Tensor | None = None,
    generator: torch.Generator | None = None,
    device=None,
    mode: str = "euler",
) -> torch.Tensor:
    """Sample the learned flow from t=0 (noise) to t=1 (data).

    ``predict_velocity`` maps (x, t, condition) -> v̂, where condition is the
    pre-fused state vector; the loop only integrates (Euler or other modes)
    and never touches network weights.

    Euler: ``x ← x + δ·v̂`` on the ascending linspace(0, 1, steps+1) grid,
    δ = 1/steps at the left-endpoint t. ``eps=None`` draws N(0, I);
    explicit ``eps`` (e.g. zeros) bypasses randomness. Inference-only.
    """
    if mode != "euler":
        raise ValueError(f"mode={mode!r} unknown; expected 'euler'")
    grid = torch.linspace(0.0, 1.0, num_inference_steps + 1)
    if eps is None:
        eps = torch.randn(*shape, generator=generator)
        if device is not None:
            eps = eps.to(device)
    x = eps
    with torch.inference_mode():
        # ------- integrate velocity field from noise to data -------
        for i in range(num_inference_steps):
            v_hat = predict_velocity(x, grid[i], condition)
            x = x + (grid[i + 1] - grid[i]) * v_hat
    return x

In [ ]:
%%writefile src/sjfpo/gae.py
"""Chunk-level semi-MDP GAE + returns (pure numpy fp64).

Each executed chunk is one decision epoch. For chunk k with length L_k:
    R_k     = Σᵢ γⁱ·r_{k,i}            (executed steps only)
    δ_k     = R_k + γ^{L_k}·V_{k+1}·(1−terminated_k) − V_k
    A_k     = δ_k + (γλ)^{L_k}·(1−terminated_k)·A_{k+1}
    G_k     = A_k + V_k
terminated cuts bootstrap and recursion; truncated (TimeLimit) bootstraps δ
but cuts the recursion into the post-reset episode. Advantages are centered
afterwards; returns are not.
"""

from __future__ import annotations

import numpy as np

__all__ = ["compute_advantages", "compute_returns", "normalize_advantages"]


def as_chunk_arrays(rewards, values, next_values, executed_n_action_steps, terminated):
    rewards = np.asarray(rewards, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    executed_n_action_steps = np.asarray(executed_n_action_steps, dtype=np.int64)
    terminated = np.asarray(terminated, dtype=bool)
    nv = np.asarray(next_values, dtype=np.float64)
    n = values.shape[0]
    if nv.ndim == 0:
        if n > 1:
            nv = np.concatenate([values[1:], nv.reshape(1)])
        else:
            nv = nv.reshape(1)
    return rewards, values, nv, executed_n_action_steps, terminated, n


def validate_chunk_stream(
    rewards, values, next_values, executed_n_action_steps, terminated, truncated, n
):
    if (
        next_values.shape != (n,)
        or executed_n_action_steps.shape != (n,)
        or terminated.shape != (n,)
        or truncated.shape != (n,)
    ):
        raise ValueError(
            f"shape mismatch: values {values.shape}, next_values {next_values.shape}, "
            f"executed_n_action_steps {executed_n_action_steps.shape}, terminated {terminated.shape}, "
            f"truncated {truncated.shape}"
        )
    if executed_n_action_steps.ndim != 1 or n == 0:
        raise ValueError("need a non-empty 1-D stream of chunks")
    if (executed_n_action_steps <= 0).any():
        raise ValueError("every chunk must have >= 1 executed step")
    if int(executed_n_action_steps.sum()) != rewards.shape[0]:
        raise ValueError(
            f"rewards has {rewards.shape[0]} steps but sum(executed_n_action_steps)={int(executed_n_action_steps.sum())}"
        )


def compute_returns(advantages, values):
    return np.asarray(advantages, dtype=np.float64) + np.asarray(
        values, dtype=np.float64
    )


def normalize_advantages(advantages, clamp: float | None = 100.0):
    advantages = np.asarray(advantages, dtype=np.float64)
    normalized = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    if clamp is not None:
        # OFF advantage_clamp (100, 100): insurance against pathological skew.
        normalized = np.clip(normalized, -clamp, clamp)
    return normalized


def compute_advantages(
    rewards,
    values,
    next_values,
    executed_n_action_steps,
    terminated,
    gamma: float = 0.99,
    lam: float = 0.95,
    normalize: bool = True,
    truncated=None,
):
    """Semi-MDP chunk-level GAE: (advantages [N], returns [N]) float64.

    next_values [N] is the bootstrap candidate (0 for terminated chunks).
    executed_n_action_steps [N] holds EXECUTED steps per chunk. truncated [N] (or
    None) marks TimeLimit hits. Returns = raw advantages + V.
    """
    rewards, values, next_values, executed_n_action_steps, terminated, n = as_chunk_arrays(
        rewards, values, next_values, executed_n_action_steps, terminated
    )
    if truncated is None:
        truncated = np.zeros(n, dtype=bool)
    else:
        truncated = np.asarray(truncated, dtype=bool)
    validate_chunk_stream(
        rewards, values, next_values, executed_n_action_steps, terminated, truncated, n
    )

    offsets = np.concatenate(([0], np.cumsum(executed_n_action_steps)))
    # ------- intra-chunk discounted rewards -------
    r_chunk = np.empty(n, dtype=np.float64)
    for k in range(n):
        steps = rewards[offsets[k] : offsets[k + 1]]
        r_chunk[k] = float(
            np.dot(gamma ** np.arange(steps.shape[0], dtype=np.float64), steps)
        )

    # ------- backward GAE recursion over chunks -------
    advantages = np.empty(n, dtype=np.float64)
    next_advantage = 0.0
    for k in range(n - 1, -1, -1):
        length = int(executed_n_action_steps[k])
        if terminated[k]:
            delta, continuation = r_chunk[k] - values[k], 0.0
        else:
            delta = r_chunk[k] + (gamma**length) * next_values[k] - values[k]
            continuation = 0.0 if truncated[k] else (gamma * lam) ** length
        advantages[k] = delta + continuation * next_advantage
        next_advantage = advantages[k]

    returns = compute_returns(advantages, values)
    if normalize:
        advantages = normalize_advantages(advantages)
    return advantages, returns

In [ ]:
%%writefile src/sjfpo/modeling_sjfpo.py
"""SJFPO policy (LeRobot-native): flow head + state conditioner + critic + obs stats.

Batch contract for ``forward`` (fp32, on policy device): observation.state
[B, T, obs], action [B, H, action_dim] real scaled, action_is_pad [B, H]
bool (absent = all executed), advantages/returns [B] (frozen GAE),
values [B] optional (v_old at collection for clipped value loss).
"""

from __future__ import annotations

import copy
from collections import deque
from typing import Any

import einx
import numpy as np
import torch

from lerobot.policies import PreTrainedPolicy
from lerobot.processor.converters import to_tensor
from lerobot.utils.constants import ACTION, DONE, OBS_STATE, REWARD, TRUNCATED

from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.collector import collect_chunks, collect_chunks_vec
from sjfpo.dct_utils import dct, idct
from sjfpo.fpo_core import (
    compute_aspo_surrogate,
    compute_cfm_loss,
    compute_policy_ratio,
    run_mc_simulation,
    sample_action,
    sample_noise,
    sample_timesteps,
)
from sjfpo.gae import compute_advantages
from sjfpo.utils import autocast_context, filter_low_freq, to_device_async, to_numpy_1d, to_tensor_1d
from sjfpo.networks import (
    Critic,
    RunningMeanStd,
    SJFPOFlowModel,
    StateConditioner,
)
from sjfpo.processor_sjfpo import make_sjfpo_pre_post_processors


class SJFPOAborted(RuntimeError):
    """Non-finite training loss (message names the offending term)."""

    def __init__(self, message: str):
        super().__init__(message)
        self.message = message


class SJFPOPolicy(PreTrainedPolicy):
    config_class = SJFPOConfig  # must match the string in @register_subclass
    name = "sjfpo"

    def __init__(
        self, config: SJFPOConfig, dataset_stats: dict[str, Any] | None = None
    ):
        super().__init__(config, dataset_stats)
        config.validate_features()
        obs_dim = int(config.robot_state_feature.shape[0])
        action_dim = int(config.action_feature.shape[0])
        self.actor = SJFPOFlowModel(
            action_dim,
            hidden_dim=config.hidden_dim,
        )
        # Fused state conditioning (Flamingo-style: state queries DCT bins),
        # computed externally; the flow head only sees the [B, D] vector.
        self.conditioner = StateConditioner(
            obs_dim,
            hidden_dim=config.hidden_dim,
        )
        self.critic = Critic(
            obs_dim,
            hidden_dim=config.hidden_dim,
            n_obs_steps=config.n_obs_steps,
        )
        self.obs_normalizer = RunningMeanStd(obs_dim)
        # Real action bounds (env.action_space); predict/select return REAL
        # actions, training batches stay scaled [-1, 1]. Non-persistent so
        # old checkpoints load unchanged; build_policy sets them per run.
        self.register_buffer("action_low", torch.full((action_dim,), -1.0), persistent=False)
        self.register_buffer("action_high", torch.full((action_dim,), 1.0), persistent=False)
        # Frozen behavior snapshot (PPO π_old): unit ratios at init, refreshed
        # per collection round; rides state_dict but excluded from the optimizer.
        # Inference-only (requires_grad off): nothing ever backprops through
        # it, and DDP treats grad-less requires_grad params as an error.
        self.old_actor = copy.deepcopy(self.actor)
        self.old_actor.requires_grad_(False)
        self.old_conditioner = copy.deepcopy(self.conditioner)
        self.old_conditioner.requires_grad_(False)
        self._frame_buffer: deque[torch.Tensor] = deque(maxlen=config.n_obs_steps)
        self._action_buffer: deque[torch.Tensor] = deque()
        self.step_count = 0
        # LeRobot-fashion pre/post pair sharing the live normalizer + bounds.
        self.preprocessor, self.postprocessor = make_sjfpo_pre_post_processors(self)
        del dataset_stats  # stats stay online (RunningMeanStd), never frozen

    @staticmethod
    def compute_target_velocity(a: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
        """v* = a − ε broadcast over MC draws: [B, *flow] x [B, m, *flow] -> [B, m, *flow]."""
        return a.unsqueeze(1) - eps

    def compute_predicted_velocity(self, x_t, tau, states, old=False) -> torch.Tensor:
        """Velocity from the live head, or the frozen snapshot when old."""
        conditioner = self.old_conditioner if old else self.conditioner
        net = self.old_actor if old else self.actor
        return net(x_t, tau, conditioner(states))

    def reset(self):
        """Reset per-episode state. Called by lerobot-eval at the start of each episode."""
        self._frame_buffer.clear()
        self._action_buffer.clear()

    def set_action_bounds(self, low: np.ndarray | torch.Tensor, high: np.ndarray | torch.Tensor) -> None:
        """Real env bounds for unscaling; defaults (±1) are identity."""
        # copy_ keeps buffer registration (device moves still follow .to()).
        self.action_low.copy_(to_tensor_1d(low, dtype=torch.float32, device=self.action_low.device))
        self.action_high.copy_(to_tensor_1d(high, dtype=torch.float32, device=self.action_high.device))

    def snapshot_behavior_policy(self) -> None:
        """Freeze current weights as the behavior policy for l_old.

        Call once per collection round, before any update on the new batch.
        """
        self.old_actor = copy.deepcopy(self.actor)
        self.old_actor.requires_grad_(False)  # deepcopy inherits live grad flags
        self.old_conditioner = copy.deepcopy(self.conditioner)
        self.old_conditioner.requires_grad_(False)

    def get_optim_params(self) -> dict:
        """Actor (decay / no-decay) + conditioner + critic groups.

        Single LR everywhere (PPO standard); snapshots never train. Zero-init
        layers (adaLN regressors incl. adaLN_out, final out) skip weight decay
        so decay never fights the identity-at-init design.
        """
        decay, no_decay = [], []
        for name, p in self.actor.named_parameters():
            # Strip the torch.compile wrapper prefix so compiled/eager group
            # identically (else save/load optimizer states stop matching).
            short = name.split("_orig_mod.")[-1]
            if "adaLN" in short or short.startswith("out."):
                no_decay.append(p)
            else:
                decay.append(p)
        return {
            "params": [
                {"params": decay},
                {"params": no_decay, "weight_decay": 0.0},
                {"params": list(self.conditioner.parameters())},
                {"params": list(self.critic.parameters())},
            ]
        }

    def standardize_obs_tensor(self, states: torch.Tensor) -> torch.Tensor:
        """Coerce [B, T, obs] / [B, obs] / [T, obs] to a [B, T, obs] window."""
        states = to_tensor(states, dtype=torch.float32)
        if states.dim() == 3:
            return states
        if states.dim() == 2:
            if states.shape[0] == self.config.n_obs_steps:
                return einx.id("t d -> 1 t d", states)
            return einx.id("d -> 1 t d", states, t=self.config.n_obs_steps)
        raise ValueError(
            f"observation.state must be [B, T, obs], [T, obs] or [B, obs]; got {tuple(states.shape)}"
        )

    def predict_action_chunk(
        self, batch: dict[str, torch.Tensor], **kwargs
    ) -> torch.Tensor:
        """Full action chunk [B, H, action_dim]; ``noise=True`` samples, else zeros."""
        noise = kwargs.get("noise", False)
        cfg = self.config
        # ------- standardize + normalize obs window -------
        states = self.standardize_obs_tensor(batch[OBS_STATE])
        states = to_device_async(states, next(self.actor.parameters()).device)
        states = self.preprocessor({OBS_STATE: states})[OBS_STATE]
        top_k_spectral_action_tokens = max(1, round(cfg.top_p_spectral_tokens * cfg.action_horizon))
        action_dim = int(self.config.action_feature.shape[0])
        with torch.inference_mode(), autocast_context(states.device):
            # ------- fuse state into one condition vector -------
            condition = self.conditioner(states)  # [B, D] fused outside the flow head
            shape = (states.shape[0], top_k_spectral_action_tokens, action_dim)
            eps = None if noise else torch.zeros(*shape, device=states.device)
            num_inference_steps = cfg.num_euler_steps if noise else cfg.num_inference_steps
            # ------- denoise action chunk with Euler sampler -------
            x0 = sample_action(
                lambda x, t, c: self.actor(x, t, c),
                shape,
                condition,
                num_inference_steps=num_inference_steps,
                eps=eps,
                device=states.device,
            )
            x0 = x0.float()
        # ------- pad + decode chunk back to action space -------
        x0 = torch.nn.functional.pad(x0, (0, 0, 0, cfg.action_horizon - top_k_spectral_action_tokens))
        scaled = idct(x0, dim=1).clamp(-1.0, 1.0)
        return self.postprocessor({ACTION: scaled})[ACTION]

    def select_action(self, batch: dict[str, torch.Tensor], **kwargs) -> torch.Tensor:
        """One action per call: replan a chunk, drain its executed prefix open-loop."""
        obs = to_tensor(batch[OBS_STATE], dtype=torch.float32)
        if obs.dim() == 1:
            if not self._frame_buffer:
                for _ in range(self.config.n_obs_steps):
                    self._frame_buffer.append(obs.clone())
            else:
                self._frame_buffer.append(obs)
            window = torch.stack(tuple(self._frame_buffer), dim=0)  # [T, obs]
        elif obs.dim() == 2:
            window = obs
        else:
            raise ValueError(
                f"select_action expects [obs] or [T, obs]; got {tuple(obs.shape)}"
            )
        if not self._action_buffer:
            chunk = self.predict_action_chunk({OBS_STATE: window}, **kwargs)
            n_execute = max(1, min(self.config.n_action_steps, chunk.shape[1]))
            for i in range(n_execute - 1, -1, -1):
                self._action_buffer.append(chunk[0, i])
        return self._action_buffer.popleft()

    def generate_rollout(
        self,
        env,
        total_steps: int,
        *,
        action_low: float = -1.0,
        action_high: float = 1.0,
        noisy: bool = True,
    ) -> dict[str, torch.Tensor]:
        """Roll out open-loop chunks; return a forward-ready batch.

        Full-horizon noisy rollouts, V(s) under frozen stats, chunk GAE.
        Obs stats stay frozen here; ``forward`` updates them in train mode.
        """
        if total_steps <= 0:
            raise ValueError(f"total_steps={total_steps} must be positive")
        cfg = self.config
        device = next(self.parameters()).device

        def propose(state: torch.Tensor) -> torch.Tensor:
            chunk = self.predict_action_chunk(
                {OBS_STATE: to_device_async(state, device)}, noise=noisy
            )[0]
            return chunk.cpu()  # env stepping stays on CPU

        def propose_batch(states: torch.Tensor) -> torch.Tensor:
            chunk = self.predict_action_chunk(
                {OBS_STATE: to_device_async(states, device)}, noise=noisy
            )
            return chunk.cpu()

        if int(getattr(env, "num_envs", 1)) > 1:
            # ------- collect open-loop chunks from env -------
            chunks = collect_chunks_vec(
                env,
                propose_batch,
                total_steps,
                action_low,
                action_high,
                n_obs_steps=cfg.n_obs_steps,
                n_action_steps=cfg.action_horizon,
            )
        else:
            # ------- collect open-loop chunks from env -------
            chunks = collect_chunks(
                env,
                propose,
                total_steps,
                action_low,
                action_high,
                n_obs_steps=cfg.n_obs_steps,
                n_action_steps=cfg.action_horizon,
            )
        executed_n_action_steps = to_tensor(
            [int(c["executed_n_action_steps"]) for c in chunks], dtype=torch.long
        )  # [N] CPU
        # Stacked once: reused for critic scoring below and the return dict
        # (stacking must stay outside inference_mode so forward() autograd
        # can consume the batch).
        states = to_device_async(
            torch.stack([c[OBS_STATE] for c in chunks]), device
        )
        next_states = to_device_async(
            torch.stack([c["next.observation.state"] for c in chunks]), device
        )
        with torch.inference_mode():
            states_n = self.preprocessor({OBS_STATE: states})[OBS_STATE]
            next_states_n = self.preprocessor({OBS_STATE: next_states})[OBS_STATE]
            # ------- score states with frozen V(s) critic -------
            with autocast_context(device):
                values = self.critic(states_n).float()
                v_next_all = self.critic(next_states_n).float()
            terminated = to_tensor([bool(c[DONE]) for c in chunks], dtype=torch.bool, device=device)
            next_values = torch.where(terminated, 0.0, v_next_all)
        all_rewards = np.concatenate([to_numpy_1d(c[REWARD]) for c in chunks])
        # ------- chunk GAE into advantages + returns -------
        advantages, returns = compute_advantages(
            all_rewards,
            to_numpy_1d(values),
            to_numpy_1d(next_values),
            to_numpy_1d(executed_n_action_steps).astype(np.int64),
            np.array([bool(c[DONE]) for c in chunks]),
            gamma=cfg.gamma,
            lam=cfg.gae_lambda,
            normalize=True,
            truncated=np.array([bool(c[TRUNCATED]) for c in chunks]),
        )
        is_pad = torch.arange(cfg.action_horizon).unsqueeze(0) >= to_tensor(
            executed_n_action_steps, dtype=torch.long
        ).unsqueeze(1)
        n_chunks = len(chunks)
        action_dim = int(self.config.action_feature.shape[0])
        # ------- common random numbers: one shared (tau, eps) draw per round -------
        # Stored in the batch and reused by every forward() on it, so l_old and
        # l_new always compare the same MC samples (OFF rollout storage). The
        # behavior snapshot is frozen per round (train loop), never per step.
        top_k = max(1, round(cfg.top_p_spectral_tokens * cfg.action_horizon))
        tau = sample_timesteps(
            cfg.num_mc_samples, (cfg.min_timestep, cfg.max_timestep), device=device
        )
        eps = sample_noise((n_chunks, cfg.num_mc_samples, top_k, action_dim), device=device)
        actions = to_device_async(torch.stack([c["action"] for c in chunks]), device)
        return {
            OBS_STATE: states,
            ACTION: actions,
            "action_is_pad": to_device_async(is_pad, device),
            "advantages": to_device_async(
                to_tensor(advantages, dtype=torch.float32), device
            ),
            "returns": to_device_async(to_tensor(returns, dtype=torch.float32), device),
            "values": values.detach().to(device),
            "tau": tau.unsqueeze(0).expand(n_chunks, -1).contiguous(),
            "eps": eps,
        }

    def forward(
        self, batch: dict[str, torch.Tensor], reduction: str = "mean"
    ) -> tuple[torch.Tensor, dict | None]:
        """Compute the combined training loss (ASPO policy + value + KL) from a rollout batch.

        Inputs (batch dict from ``generate_rollout``):
            observation.state: [B, T, obs_dim]
            action:              [B, H, action_dim]
            advantages:          [B]  (frozen GAE; never optimized)
            returns:             [B]  (GAE targets for value loss)
            values:              [B]  (optional; frozen v_old for clipped value loss)
            action_is_pad:       [B, H] bool (accepted but ignored: the CFM loss
                always sees the full proposed chunk, paper-original)
            tau:                 [B, m] (stored rollout draw; reused for CRN)
            eps:                 [B, m, K, act] (stored rollout draw)

        Outputs:
            (loss, info_dict) where ``loss`` is scalar if ``reduction="mean"``
            or per-chunk [B] if ``reduction="none"``. ``info_dict`` holds detached
            scalar summaries: ``policy_loss``, ``value_loss``,
            ``explained_var``, ``mean_log_ratio``, plus diagnostics (auto-logged to wandb as
            ``train/...``): ``flow_rho_mean/std``, ``flow_v_star_rms``, and
            ``cond_*`` RMS norms (``temb``, ``pooled``, ``c``, per-block mean
            ``gate_msa/mlp`` and ``1+scale``) — gates near 0 mean the spectrum
            is unused (expected early under adaLN-Zero).

        Gradient behavior:
            - ``actor`` and ``critic`` receive gradients.
            - ``old_actor`` (behavior snapshot) is frozen (``inference_mode``).
        """
        cfg = self.config
        states = to_tensor(batch[OBS_STATE], dtype=torch.float32)
        actions = to_tensor(batch[ACTION], dtype=torch.float32)
        advantages = to_tensor(batch["advantages"], dtype=torch.float32)
        returns = to_tensor(batch["returns"], dtype=torch.float32)  # [B]
        if states.shape[-2] != cfg.n_obs_steps:
            raise ValueError(
                f"state length {states.shape[-2]} != n_obs_steps {cfg.n_obs_steps}"
            )
        if actions.shape[-2] != cfg.action_horizon:
            raise ValueError(
                f"action action_horizon {actions.shape[-2]} != action_horizon {cfg.action_horizon}"
            )
        if self.training:
            self.preprocessor.update_stats(
                states
            )  # train batches only; rollout/eval frozen
        states = self.preprocessor({OBS_STATE: states})[OBS_STATE]
        top_k_spectral_action_tokens = max(1, round(cfg.top_p_spectral_tokens * cfg.action_horizon))
        # ------- keep K-lowest action DCT bins, drop the rest -------
        # Paper-original: the loss sees the FULL proposed chunk. Unexecuted
        # tails (early termination / budget cutoff) only shape GAE via
        # executed_n_action_steps; they are never zero-masked here (masking
        # before the DCT leaks zeros into every frequency bin).
        spectral_action_tokens = filter_low_freq(dct(actions, dim=1), top_k_spectral_action_tokens, dim=1)
        n = states.shape[0]
        # ------- MC draws: reuse the rollout's stored (tau, eps) when present ---
        # (common random numbers: l_old/l_new compare identical samples across
        # all epochs, OFF rollout storage). Batches built by hand (tests) fall
        # back to a fresh draw.
        if "tau" in batch and "eps" in batch:
            tau = to_tensor(batch["tau"], dtype=torch.float32, device=states.device)
            tau = tau[0] if tau.dim() == 2 else tau
            eps = to_tensor(batch["eps"], dtype=torch.float32, device=states.device)
            if tuple(eps.shape) != (
                n,
                cfg.num_mc_samples,
                top_k_spectral_action_tokens,
                actions.shape[-1],
            ):
                raise ValueError(
                    f"stored eps shape {tuple(eps.shape)} != "
                    f"({(n, cfg.num_mc_samples, top_k_spectral_action_tokens, actions.shape[-1])})"
                )
        else:
            tau = sample_timesteps(
                cfg.num_mc_samples,
                (cfg.min_timestep, cfg.max_timestep),
                device=states.device,
            )
            eps = sample_noise(
                (n, cfg.num_mc_samples, top_k_spectral_action_tokens, actions.shape[-1]),
                device=states.device,
            )

        # ------- interpolate noisy anchors along the flow -------
        tau_bn, eps_flat, a_rep, flow_shape, num_mc = run_mc_simulation(
            spectral_action_tokens, eps, tau
        )
        x_t_flat = (
            (1.0 - tau_bn[:, None]) * eps_flat + tau_bn[:, None] * a_rep
        ).reshape(-1, *flow_shape)  # (1−t)·ε+t·a
        # ------- fuse states once, replicate over MC draws -------
        cond_flat = einx.id(
            "n m d -> (n m) d", einx.id("n d -> n m d", self.conditioner(states), m=num_mc)
        )

        # ------- live velocities vs frozen snapshot velocities -------
        # (tau_bn already holds the flattened MC times; no second expansion.)
        with torch.inference_mode(), autocast_context(states.device):
            old_cond_flat = einx.id(
                "n m d -> (n m) d",
                einx.id("n d -> n m d", self.old_conditioner(states), m=num_mc),
            )
            v_old = self.old_actor(x_t_flat, tau_bn, old_cond_flat).float()
        with autocast_context(states.device):
            v_new = self.actor(x_t_flat, tau_bn, cond_flat).float()
        cond_stats = dict(self.actor.last_cond_stats)
        # ------- CFM losses under target velocities -------
        v_star = self.compute_target_velocity(spectral_action_tokens, eps)
        v_old_r = v_old.reshape(n, num_mc, *flow_shape)
        v_new_r = v_new.reshape(n, num_mc, *flow_shape)
        # ------- ASPO trust-region policy loss (paper Eq.12-13, OFF clamps) ----
        adv_mc = advantages.unsqueeze(1)
        rho_bin = None
        eps_bin = None
        if cfg.use_per_bin_clip:
            # Phase 1 per-bin trust region: mean-of-psi over DCT bins with a
            # two-group epsilon (tight on low-frequency intent bins, loose on
            # high-frequency detail). Same elementwise ratio/ASPO math, no
            # fpo_core change; OFF-faithful scalar path when disabled.
            l_old_bin = compute_cfm_loss(v_old_r, v_star, reduce_mode="bin")
            l_new_bin = compute_cfm_loss(v_new_r, v_star, reduce_mode="bin")
            rho_bin = compute_policy_ratio(l_old_bin, l_new_bin)
            n_bins = rho_bin.shape[-1]
            split = max(1, n_bins // 2)
            eps_bin = torch.full(
                (n_bins,), cfg.epsilon_clip_highfreq, device=rho_bin.device
            )
            eps_bin[:split] = cfg.epsilon_clip_lowfreq
            psi = compute_aspo_surrogate(rho_bin, adv_mc.unsqueeze(-1), eps_bin)
            policy_per_chunk = -psi.mean(dim=(1, 2))
        l_old = compute_cfm_loss(v_old_r, v_star, reduce_mode="sample")
        l_new = compute_cfm_loss(v_new_r, v_star, reduce_mode="sample")
        rho = compute_policy_ratio(l_old, l_new)
        if rho_bin is None:  # scalar path owns the loss; per-bin logged only
            psi = compute_aspo_surrogate(
                rho, adv_mc, cfg.epsilon_clip
            )
            policy_per_chunk = -psi.mean(dim=1)
        # ------- value loss on raw returns (OFF: no 0.5, unclipped default) ---
        with autocast_context(states.device):
            v_pred = self.critic(states).float()
        # Regress RAW returns (no per-batch whitening; the target must not
        # depend on its minibatch). OFF disables value clipping by default
        # ("value clipping adds instability" under tight policy clip).
        returns_d = returns.to(v_pred.device)
        se = (v_pred - returns_d).pow(2)
        if cfg.use_clipped_value_loss and "values" in batch:
            old_values = to_tensor(
                batch["values"], dtype=torch.float32, device=v_pred.device
            )
            v_clipped = old_values + (v_pred - old_values).clamp(
                -cfg.epsilon_clip, cfg.epsilon_clip
            )
            value_per_chunk = torch.max(se, (v_clipped - returns_d).pow(2))
        else:
            value_per_chunk = se
        if reduction == "mean":
            policy_loss = policy_per_chunk.mean()
            value_loss = value_per_chunk.mean()
            loss = policy_loss + cfg.vf_coef * value_loss
        elif reduction == "none":
            loss = policy_per_chunk + cfg.vf_coef * value_per_chunk
        else:
            raise ValueError(f"reduction={reduction!r} must be 'mean' or 'none'")
        if not bool(torch.isfinite(loss).all()):
            raise SJFPOAborted(
                f"Non-finite loss: policy={float(policy_per_chunk.mean().detach())}, "
                f"value={float(value_per_chunk.mean().detach())}"
            )
        with torch.inference_mode():
            mean_log_ratio = float((l_old - l_new).mean())
            rho_mean = float(rho.detach().mean())
            rho_std = float(rho.detach().std(unbiased=False))
            v_star_rms = float(v_star.detach().pow(2).mean().sqrt())
            # PPO standard critic health: 1 - Var(G - V) / Var(G).
            returns_var = returns_d.detach().var(unbiased=False).clamp_min(1e-8)
            explained_var = float(
                1.0 - (returns_d.detach() - v_pred.detach()).var(unbiased=False) / returns_var
            )
            # Phase 0 per-bin trust diagnostics (read-only on the scalar path;
            # live trust region when use_per_bin_clip): same elementwise
            # ratio math, unreduced over bins. Reuses the loss path's rho_bin
            # when present so logged stats match the applied trust region.
            if rho_bin is None:
                l_old_bin = compute_cfm_loss(v_old_r, v_star, reduce_mode="bin")
                l_new_bin = compute_cfm_loss(v_new_r, v_star, reduce_mode="bin")
                rho_bin = compute_policy_ratio(l_old_bin, l_new_bin)
            rho_bin_mean = rho_bin.mean(dim=(0, 1))  # [K] over batch, mc
            n_bins = rho_bin_mean.numel()
            split = max(1, n_bins // 2)
            rho_bin_range = float(rho_bin_mean.max() - rho_bin_mean.min())
            rho_bin_low = float(rho_bin_mean[:split].mean())
            rho_bin_high = float(
                rho_bin_mean[split:].mean()
                if n_bins > 1
                else rho_bin_mean.mean()
            )
            rho_bin_outside = float(
                (
                    (rho_bin < 1.0 - cfg.epsilon_clip)
                    | (rho_bin > 1.0 + cfg.epsilon_clip)
                )
                .float()
                .mean()
            )
            # Per-group clip fractions against the eps actually applied
            # (per-bin eps when enabled, scalar eps for both groups otherwise).
            eps_low = cfg.epsilon_clip_lowfreq if eps_bin is not None else cfg.epsilon_clip
            eps_high = cfg.epsilon_clip_highfreq if eps_bin is not None else cfg.epsilon_clip
            rho_bin_low_outside = float(
                (
                    (rho_bin[..., :split] < 1.0 - eps_low)
                    | (rho_bin[..., :split] > 1.0 + eps_low)
                )
                .float()
                .mean()
            )
            rho_bin_high_outside = (
                float(
                    (
                        (rho_bin[..., split:] < 1.0 - eps_high)
                        | (rho_bin[..., split:] > 1.0 + eps_high)
                    )
                    .float()
                    .mean()
                )
                if n_bins > 1
                else rho_bin_low_outside
            )
        return loss, {
            "policy_loss": float(policy_per_chunk.mean().detach()),
            "value_loss": float(value_per_chunk.mean().detach()),
            "explained_var": explained_var,
            "mean_log_ratio": mean_log_ratio,
            "flow_rho_mean": rho_mean,
            "flow_rho_std": rho_std,
            "flow_rho_bin_range": rho_bin_range,
            "flow_rho_bin_lowfreq": rho_bin_low,
            "flow_rho_bin_highfreq": rho_bin_high,
            "flow_rho_bin_outside_frac": rho_bin_outside,
            "flow_rho_bin_low_outside_frac": rho_bin_low_outside,
            "flow_rho_bin_high_outside_frac": rho_bin_high_outside,
            "flow_v_star_rms": v_star_rms,
            **{f"cond_{k}": float(v.detach()) for k, v in cond_stats.items()},
        }

In [ ]:
%%writefile src/sjfpo/networks.py
"""Networks: window encoders, state conditioner (raw + spectral), flow heads, V(s) critic.

Tensor conventions: B batch, N chunk batch, T obs window, H action_horizon,
K spectral tokens, D hidden dim, action_dim env action dim. Params are fp32;
compute may autocast matmuls (see utils.autocast_context).
"""
from __future__ import annotations

import math

import einx
import torch
from torch import nn

from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.dct_utils import dct

# Single source for constructor defaults (see SJFPOConfig).
_DEFAULTS = SJFPOConfig()


class SwiGLU(nn.Module):
    """Gated SiLU feedforward: down(silu(gate(x)) * up(x)).

    Drop-in for ``Sequential(Linear(in_dim, hidden_dim), SiLU,
    Linear(hidden_dim, out_dim))`` with the same (in, hidden, out) widths.
    """

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.gate = nn.Linear(in_dim, hidden_dim)
        self.up = nn.Linear(in_dim, hidden_dim)
        self.down = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """[..., in_dim] -> [..., out_dim]."""
        return self.down(nn.functional.silu(self.gate(x)) * self.up(x))


class CondMLP(nn.Module):
    """SiLU MLP for conditioning regression (DiT canon: SiLU then Linear).

    Gating belongs in block FFNs, not in the gate-regression path; the
    modulation readout stays (near-)linear. Final layer is zero-initialized
    by callers that need identity-at-init (adaLN-Zero)."""

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """[..., in_dim] -> [..., out_dim]."""
        return self.fc2(self.act(self.fc1(x)))


def resolve_config_value(name: str, override):
    """Policy default for ``name`` unless ``override`` is not None."""
    return override if override is not None else getattr(_DEFAULTS, name)

class SinusoidalEncoding(nn.Module):
    """Fixed sinusoidal codes (stateless: nothing in state_dict).

    One helper for both uses: position codes (``scale=1``) and flow-time
    codes (``scale=timestep_scale``, resolving the DDPM ladder designed for
    integer steps 0-1000 when ``t`` lives in [0, 1]).
    """

    def __init__(self, dim: int, scale: float = 1.0):
        super().__init__()
        if dim <= 0:
            raise ValueError(f"dim={dim} must be positive")
        self.dim = dim
        self.scale = float(scale)

    def frequencies(self, half: int, device) -> torch.Tensor:
        return torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=device, dtype=torch.float32)
            / max(half - 1, 1)
        )

    def add_to(self, x: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
        """Add codes for ``positions`` ([..., L]) to ``x`` ([..., L, D])."""
        scale = 1.0 if self.scale is None else float(self.scale)
        ang = (positions.to(dtype=torch.float32) * scale).unsqueeze(-1) * self.frequencies(
            self.dim // 2, x.device
        )
        pe = torch.cat([ang.sin(), ang.cos()], dim=-1)
        if pe.shape[-1] < self.dim:  # odd dim: pad the missing column with zeros
            pe = torch.nn.functional.pad(pe, (0, self.dim - pe.shape[-1]))
        return x + pe.to(dtype=x.dtype)

    def encode_times(self, t: torch.Tensor) -> torch.Tensor:
        """Sinusoidal embedding for continuous timestep values.

        Returns an embedding of shape ``t.shape + (dim,)``; ``t`` is scaled
        first (flow times in [0, 1] need the DDPM-scale ladder to resolve).
        """
        t = torch.as_tensor(t, dtype=torch.float32) * float(self.scale)
        half = self.dim // 2
        ang = t.unsqueeze(-1) * self.frequencies(half, t.device)
        emb = torch.cat([ang.sin(), ang.cos()], dim=-1)
        if self.dim % 2:
            emb = torch.nn.functional.pad(emb, (0, 1))
        return emb

    def positions(self, length: int, device) -> torch.Tensor:
        """0..L-1 index vector for slot/frame codes."""
        return torch.arange(length, dtype=torch.float32, device=device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add position codes by slot index: [..., L, D] -> [..., L, D]."""
        return self.add_to(x, self.positions(x.shape[-2], x.device))

class SelfAttention(nn.Module):
    """Multi-head self-attention on ``F.scaled_dot_product_attention``.

    Fused QKV + output projections; flash / memory-efficient kernels are
    dispatched automatically (and compose with ``torch.compile``), unlike the
    unfused ``nn.MultiheadAttention`` path. ``key_mask`` follows the familiar
    ``key_padding_mask`` convention ([B, L] bool, True = ignore), inverted
    internally since SDPA boolean masks mean "attend".
    """

    def __init__(self, hidden_dim: int, n_heads: int = 4):
        super().__init__()
        if hidden_dim % n_heads != 0:
            raise ValueError(f"hidden_dim={hidden_dim} must split over {n_heads} heads")
        self.n_heads = n_heads
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out = nn.Linear(hidden_dim, hidden_dim)

    def forward(
        self, x: torch.Tensor, key_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """[B, L, D] -> [B, L, D]."""
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # ------- split heads (named axes beat view/transpose chains) -------
        q = einx.id("b l (h d) -> b h l d", q, h=self.n_heads)
        k = einx.id("b l (h d) -> b h l d", k, h=self.n_heads)
        v = einx.id("b l (h d) -> b h l d", v, h=self.n_heads)
        mask = None
        if key_mask is not None:
            mask = einx.id(
                "b l -> b 1 1 l",
                ~torch.as_tensor(key_mask, dtype=torch.bool, device=x.device),
            )
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out(einx.id("b h l d -> b l (h d)", y))


class WindowEncoder(nn.Module):
    """One pre-norm residual block over a time window -> Z[B, L, hidden_dim].

    Single class for state windows and action chunks: the only difference is
    the optional boolean key mask [B, L] (True = pad, ignore as attention
    key), which hides padded tails. Only the conditioner owns one now (the
    critic is a plain MLP on the current frame); call sites keep the
    ``state_encoder`` / ``action_encoder`` attribute names, so old
    checkpoints still load. One global-mixing layer suffices for short,
    smooth proprioceptive windows.
    """

    def __init__(
        self,
        in_dim: int,
        n_obs_steps: int | None = None,
        hidden_dim: int | None = None,
        n_heads: int = 4,
        mlp_ratio: float | None = None,
    ):
        super().__init__()
        n_obs_steps = resolve_config_value("n_obs_steps", n_obs_steps)
        hidden_dim = resolve_config_value("hidden_dim", hidden_dim)
        mlp_ratio = float(resolve_config_value("dit_mlp_ratio", mlp_ratio))
        self.frame_proj = nn.Linear(in_dim, hidden_dim)
        self.pos_encoding = SinusoidalEncoding(hidden_dim)
        self.norm1 = nn.RMSNorm(hidden_dim, eps=1e-6)
        self.attn1 = SelfAttention(hidden_dim, n_heads)
        self.norm2 = nn.RMSNorm(hidden_dim, eps=1e-6)
        mlp_hidden = int(hidden_dim * mlp_ratio)
        self.mlp1 = SwiGLU(hidden_dim, mlp_hidden, hidden_dim)

    def forward(self, x: torch.Tensor, key_mask: torch.Tensor | None = None) -> torch.Tensor:
        """[B, L, in_dim] -> [B, L, hidden_dim]; pad rows masked as keys when given."""
        # ------- project frames into latent space + add position codes -------
        h = self.frame_proj(x)
        h = self.pos_encoding(h)
        if key_mask is not None:
            key_mask = torch.as_tensor(key_mask, dtype=torch.bool, device=h.device)
        # ------- self-attention block (pad rows masked as keys) -------
        n = self.norm1(h)
        h = h + self.attn1(n, key_mask)
        h = h + self.mlp1(self.norm2(h))
        return h

class RunningMeanStd(nn.Module):
    """Per-dim running mean/variance over raw obs frames (buffers ride state_dict).

    update() folds raw frames in with parallel Welford; forward() normalizes
    with frozen stats. Fresh (count == 0) is the identity.
    """

    def __init__(self, dim: int, eps: float = 1e-8, clip: float = 10.0):
        super().__init__()
        if dim <= 0:
            raise ValueError(f"dim={dim} must be positive")
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("var", torch.ones(dim))
        self.register_buffer("count", torch.tensor(0.0))
        self.eps = eps
        self.clip = clip

    def update(self, x: torch.Tensor) -> None:
        """Fold raw frames into the running stats (any device, no grad)."""
        batch = x.detach().reshape(-1, x.shape[-1]).to(self.mean.device)
        b = batch.shape[0]
        if b == 0:
            return
        b_mean = batch.mean(dim=0)
        b_var = batch.var(dim=0, unbiased=False)
        delta = b_mean - self.mean
        total = self.count + b
        self.mean += delta * b / total
        self.var = (
            self.count * self.var + b * b_var + delta.square() * self.count * b / total
        ) / total
        self.count = total

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize with frozen stats; identity until the first update."""
        if int(self.count) == 0:
            return x
        norm = (x - self.mean.to(x.device)) * torch.rsqrt(
            self.var.to(x.device) + self.eps
        )
        return norm.clamp(-self.clip, self.clip)

def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """adaLN modulation: x * (1 + scale) + shift (shift/scale broadcast over tokens)."""
    return x * (1.0 + scale) + shift


class DiTBlock(nn.Module):
    """Pre-norm DiT block with adaLN-Zero conditioning.

    x <- x + gate_msa * Attn(modulate(norm(x))); x <- x + gate_mlp * MLP(...).
    Zero-initialized modulation makes each block start as the identity.
    FFN is SwiGLU (DiT uses GELU); width still follows mlp_ratio.
    """

    def __init__(self, hidden_dim: int, n_heads: int, mlp_ratio: float):
        super().__init__()
        self.norm1 = nn.RMSNorm(hidden_dim)
        self.attn = SelfAttention(hidden_dim, n_heads)
        self.norm2 = nn.RMSNorm(hidden_dim)
        mlp_hidden = int(hidden_dim * mlp_ratio)
        self.mlp = SwiGLU(hidden_dim, mlp_hidden, hidden_dim)
        self.adaLN = CondMLP(hidden_dim, hidden_dim, 6 * hidden_dim)
        nn.init.constant_(self.adaLN.fc2.weight, 0.0)
        nn.init.constant_(self.adaLN.fc2.bias, 0.0)
        self.last_gate_norms: dict[str, torch.Tensor] | None = None  # detached RMS stash

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """([B, H, D], [B, D]) -> [B, H, D]."""
        # ------- regress adaLN-Zero modulation from condition -------
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN(c)[
            :, None, :
        ].chunk(6, dim=-1)
        # ------- gated self-attention residual -------
        h = modulate(self.norm1(x), shift_msa, scale_msa)
        x = x + gate_msa * self.attn(h)
        # ------- gated MLP residual -------
        out = x + gate_mlp * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))
        self.last_gate_norms = {
            "gate_msa": gate_msa.detach().pow(2).mean().sqrt(),
            "gate_mlp": gate_mlp.detach().pow(2).mean().sqrt(),
            "scale_msa": (1.0 + scale_msa).detach().pow(2).mean().sqrt(),
            "scale_mlp": (1.0 + scale_mlp).detach().pow(2).mean().sqrt(),
        }
        return out


class StateConditioner(nn.Module):
    """Fuse raw state window + its DCT bins into one [B, D] vector.

    Mean-pool each stream, concat, one linear readout. Bin order rides a
    sinusoidal code, so the spectral stream stays frequency-indexed. (A
    query-dependent cross-attention fusion was cut: mean+concat keeps the
    spectral path at a sixth of the modules.)
    """

    def __init__(
        self,
        obs_dim: int,
        hidden_dim: int | None = None,
        n_heads: int = 4,
    ):
        super().__init__()
        if obs_dim <= 0:
            raise ValueError("obs_dim must be positive")
        hidden_dim = int(resolve_config_value("hidden_dim", hidden_dim))
        del n_heads  # accepted for call-site compat; fusion is linear now
        self.state_encoder = WindowEncoder(obs_dim, hidden_dim=hidden_dim)
        self.bin_norm = nn.RMSNorm(obs_dim)
        self.bin_encoding = SinusoidalEncoding(obs_dim)  # bin index = frequency order
        self.key_proj = nn.Linear(obs_dim, hidden_dim)
        self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        """[B, T, obs] -> [B, D]."""
        # ------- encode raw window + DCT bins, pool each stream -------
        raw = self.state_encoder(states).mean(dim=1)
        spec = self.key_proj(self.bin_encoding(self.bin_norm(dct(states, dim=1)))).mean(
            dim=1
        )
        # ------- fuse by concat + linear -------
        return self.fuse(torch.cat([raw, spec], dim=-1))


class SJFPOFlowModel(nn.Module):
    """DiT velocity model over noisy-action tokens, conditioned via adaLN.

    Takes a pre-fused condition vector [B, D] (see StateConditioner);
    ``c = time + condition`` modulates every block (adaLN-Zero). The
    transformer never sees state tokens, so there is exactly one
    observation path and no competing in-context route. Only action
    positions exist, so no readout slice.
    """

    def __init__(
        self,
        action_dim: int,
        hidden_dim: int | None = None,
        dit_depth: int | None = None,
        dit_heads: int | None = None,
        dit_mlp_ratio: float | None = None,
        dit_time_dim: int | None = None,
    ):
        super().__init__()
        hidden_dim = int(resolve_config_value("hidden_dim", hidden_dim))
        dit_depth = int(resolve_config_value("dit_depth", dit_depth))
        dit_heads = int(resolve_config_value("dit_heads", dit_heads))
        dit_mlp_ratio = float(resolve_config_value("dit_mlp_ratio", dit_mlp_ratio))
        dit_time_dim = int(resolve_config_value("dit_time_dim", dit_time_dim))
        self.time_dim = dit_time_dim
        self.timestep_embedder = SinusoidalEncoding(
            self.time_dim, scale=float(resolve_config_value("timestep_scale", None))
        )
        self.in_proj = nn.Linear(action_dim, hidden_dim)
        self.bin_encoding = SinusoidalEncoding(hidden_dim)
        self.time_mlp = CondMLP(dit_time_dim, 4 * hidden_dim, hidden_dim)
        self.blocks = nn.ModuleList(
            DiTBlock(hidden_dim, dit_heads, dit_mlp_ratio) for _ in range(dit_depth)
        )
        self.norm_out = nn.RMSNorm(hidden_dim)
        self.adaLN_out = nn.Linear(hidden_dim, 2 * hidden_dim)
        nn.init.constant_(self.adaLN_out.weight, 0.0)
        nn.init.constant_(self.adaLN_out.bias, 0.0)
        self.out = nn.Linear(hidden_dim, action_dim)
        # Zero final layer: an untrained policy predicts ~zero velocity.
        nn.init.constant_(self.out.weight, 0.0)
        nn.init.constant_(self.out.bias, 0.0)
        self.last_cond_stats: dict[str, torch.Tensor] | None = None

    def forward(self, x_t: torch.Tensor, t, condition: torch.Tensor) -> torch.Tensor:
        """Predict velocity: ([B, Ka, action_dim], t, [B, D]) -> [B, Ka, action_dim].

        Ka = top-p low-frequency action rows (<= H); condition is the
        pre-fused state vector (see StateConditioner)."""
        # ------- embed flow timestep -------
        timesteps_embedding = self.timestep_embedder.encode_times(t).to(
            device=x_t.device, dtype=x_t.dtype
        )
        if timesteps_embedding.dim() == 1:
            timesteps_embedding = timesteps_embedding.expand(x_t.shape[0], -1)
        # ------- blend time + fused state into one adaLN vector -------
        c = self.time_mlp(timesteps_embedding) + condition.to(dtype=x_t.dtype)
        # ------- embed noisy action tokens + add slot codes -------
        h = self.bin_encoding(self.in_proj(x_t))
        # ------- denoise through DiT blocks -------
        for block in self.blocks:
            h = block(h, c)
        self.last_cond_stats = {
            "temb": timesteps_embedding.detach().pow(2).mean().sqrt(),
            "fused": condition.detach().pow(2).mean().sqrt(),
            "c": c.detach().pow(2).mean().sqrt(),
            **{
                k: torch.stack(
                    [b.last_gate_norms[k] for b in self.blocks]
                ).mean()
                for k in ("gate_msa", "gate_mlp", "scale_msa", "scale_mlp")
            },
        }
        # ------- read out velocity -------
        shift_out, scale_out = self.adaLN_out(c)[:, None, :].chunk(2, dim=-1)
        return self.out(modulate(self.norm_out(h), shift_out, scale_out))

class Critic(nn.Module):
    """State-value baseline V(s) (PPO standard): MLP on the current frame.

    OFF uses an MLP critic; temporal attention is unnecessary for V(s), so
    the window encoder lives only in the conditioner now. Windows are still
    accepted ([B, T, obs]) with the last frame scored.
    """

    def __init__(
        self,
        obs_dim: int,
        hidden_dim: int | None = None,
        n_obs_steps: int | None = None,
    ):
        super().__init__()
        if obs_dim <= 0:
            raise ValueError("obs_dim must be positive")
        hidden_dim = resolve_config_value("hidden_dim", hidden_dim)
        del n_obs_steps  # accepted for call-site compat; only the last frame scores
        self.mlp = CondMLP(obs_dim, hidden_dim, 1)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        """[B, T, obs] -> [B]."""
        return self.mlp(states[:, -1, :]).squeeze(-1)

In [ ]:
%%writefile src/sjfpo/processor_sjfpo.py
"""SJFPO pre/post processors (LeRobot fashion): obs normalize, action unscale.

Mirrors other policies' ``processor_<name>.py``: named ``ProcessorStep``s plus a
``make_sjfpo_pre_post_processors`` pair factory. Two SJFPO deviations from the
LeRobot default:
- stats are ONLINE (RunningMeanStd updated every training batch), not frozen
  dataset stats, so the obs step shares the policy's live normalizer object
  and exposes ``update_stats`` (no LeRobot equivalent);
- steps operate on the policy's native batch dicts (``{"observation.state"}`` /
  ``{"action"}``) instead of EnvTransitions; shapes pass through unchanged.
"""

from __future__ import annotations

import torch

from lerobot.configs import PipelineFeatureType, PolicyFeature
from lerobot.processor.pipeline import ProcessorStep, ProcessorStepRegistry
from lerobot.utils.constants import ACTION, OBS_STATE

from sjfpo.networks import RunningMeanStd


@ProcessorStepRegistry.register(name="sjfpo_observation_normalizer")
class SJFPOObservationProcessorStep(ProcessorStep):
    """Normalize ``observation.state`` with the shared live RunningMeanStd."""

    def __init__(self, normalizer: RunningMeanStd):
        self.normalizer = normalizer

    def normalize(self, states: torch.Tensor) -> torch.Tensor:
        """Raw window -> normalized window (identity until the first update)."""
        return self.normalizer(states)

    def update_stats(self, states: torch.Tensor) -> None:
        """Fold raw training frames into the running stats (train batches only)."""
        self.normalizer.update(states)

    def __call__(
        self, batch: dict[str, torch.Tensor]
    ) -> dict[str, torch.Tensor]:
        batch = dict(batch)
        batch[OBS_STATE] = self.normalize(batch[OBS_STATE])
        return batch

    def transform_features(
        self, features: dict[PipelineFeatureType, dict[str, PolicyFeature]]
    ) -> dict[PipelineFeatureType, dict[str, PolicyFeature]]:
        return features


@ProcessorStepRegistry.register(name="sjfpo_action_unnormalizer")
class SJFPOActionPostprocessorStep(ProcessorStep):
    """Unscale policy action chunks from [-1, 1] to real env bounds."""

    def __init__(self, action_low: torch.Tensor, action_high: torch.Tensor):
        self.action_low = action_low
        self.action_high = action_high

    def unnormalize(self, scaled: torch.Tensor) -> torch.Tensor:
        """Scaled chunk -> real env actions (identity when bounds are ±1)."""
        low = self.action_low.to(scaled.device)
        high = self.action_high.to(scaled.device)
        return low + 0.5 * (scaled + 1.0) * (high - low)

    def __call__(
        self, batch: dict[str, torch.Tensor]
    ) -> dict[str, torch.Tensor]:
        batch = dict(batch)
        batch[ACTION] = self.unnormalize(batch[ACTION])
        return batch

    def transform_features(
        self, features: dict[PipelineFeatureType, dict[str, PolicyFeature]]
    ) -> dict[PipelineFeatureType, dict[str, PolicyFeature]]:
        return features


def make_sjfpo_pre_post_processors(
    policy,
) -> tuple[SJFPOObservationProcessorStep, SJFPOActionPostprocessorStep]:
    """Build the (preprocessor, postprocessor) pair sharing the policy's live state.

    Steps hold references (not copies): training stat updates and
    ``set_action_bounds`` stay visible, and ``.to(device)`` keeps working
    because the tensors remain owned by the policy module.
    """
    return (
        SJFPOObservationProcessorStep(policy.obs_normalizer),
        SJFPOActionPostprocessorStep(policy.action_low, policy.action_high),
    )

In [ ]:
%%writefile src/sjfpo/sweep.py
"""Reentrant sweep executor: scan run dirs, execute pending runs in order."""

from __future__ import annotations

import atexit
import json
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from sjfpo.config import RunSpec


class SweepError(RuntimeError):
    """Scan-time errors (corrupt manifest, missing files)."""

    pass


@dataclass(frozen=True)
class ScanResult:
    """Scan outcome: pending (missing+partial), skipped (complete), diverged."""

    pending: list[RunSpec]
    skipped: list[RunSpec]
    diverged: list[RunSpec]


def format_run_directory_name(spec: RunSpec) -> str:
    """Per-run subdir: {method}_{env}_seed{seed}/."""
    return f"{spec.method}_{spec.env}_seed{spec.seed}"


def read_run_manifest(run_dir: Path) -> dict[str, Any]:
    """Read and parse manifest.json, raising SweepError on failure."""
    manifest_path = run_dir / "manifest.json"
    try:
        return json.loads(manifest_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise SweepError(f"corrupt manifest in {run_dir}: {e}") from e
    except OSError as e:
        raise SweepError(f"failed to read manifest in {run_dir}: {e}") from e


def classify_run_status(
    run_dir: Path,
    dir_name: str,
    manifest_reader: Callable[[Path], dict[str, Any]],
) -> str:
    """Classify one run dir: missing | diverged | complete | partial."""
    if not run_dir.is_dir():
        return "missing"
    if not (run_dir / "manifest.json").is_file():
        return "missing"
    try:
        manifest = manifest_reader(run_dir)
    except SweepError:
        raise SweepError(f"failed to scan run {dir_name}") from None
    if manifest.get("diverged", False) is True:
        return "diverged"
    if manifest.get("complete", False) is True:
        return "complete"
    return "partial"


def print_session_summary(
    completed_count: int,
    total_count: int,
    session_start_time: float,
    failed: list[str] | None = None,
) -> None:
    """Print completed/failed/remaining/hours session summary."""
    elapsed_hours = (time.time() - session_start_time) / 3600.0
    failed = list(failed or [])
    remaining = total_count - completed_count - len(failed)
    print(
        f"SESSION SUMMARY: {completed_count} completed, "
        f"{len(failed)} failed, {remaining} remaining, "
        f"{elapsed_hours:.2f} hours elapsed"
    )
    for name in failed:
        print(f"SESSION SUMMARY FAILED: {name}")


def scan_runs(
    matrix: tuple[RunSpec, ...],
    base_dir: str | Path,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
    manifest_reader: Callable[[Path], dict[str, Any]] = read_run_manifest,
    sort_key: Callable[[RunSpec], tuple[Any, ...]] = lambda s: (
        s.method,
        s.env,
        s.seed,
    ),
) -> ScanResult:
    """Scan the run matrix, classifying each run as pending/skipped/diverged."""
    base = Path(base_dir)
    pending: list[RunSpec] = []
    skipped: list[RunSpec] = []
    diverged: list[RunSpec] = []
    index_map = {spec: i for i, spec in enumerate(matrix)}
    for spec in matrix:
        run_dir = base / dir_name_fn(spec)
        status = classify_run_status(run_dir, dir_name_fn(spec), manifest_reader)
        if status == "missing" or status == "partial":
            pending.append(spec)
        elif status == "complete":
            skipped.append(spec)
        else:
            diverged.append(spec)
    pending.sort(key=lambda s: sort_key(s) + (index_map.get(s, 0),))
    return ScanResult(pending=pending, skipped=skipped, diverged=diverged)


def flush_pending_runs(
    pending_runs: list[RunSpec],
    base_dir: str | Path,
    run_fn: Callable[[RunSpec, Path], dict[str, Any]],
    session_start_time: float,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
) -> tuple[list[dict[str, Any]], list[str]]:
    """Execute pending runs sequentially; failures are recorded, never silent."""
    base = Path(base_dir)
    results: list[dict[str, Any]] = []
    failed: list[str] = []
    completed_count = 0
    try:
        for spec in pending_runs:
            name = dir_name_fn(spec)
            try:
                result = run_fn(spec, base)
                results.append(result)
                completed_count += 1
            except Exception as e:
                failed.append(name)
                print(f"ERROR in run {name}: {e}")
                continue
    finally:
        print_session_summary(
            completed_count, len(pending_runs), session_start_time, failed
        )
    return results, failed


def announce_scan_results(scan_result: ScanResult) -> None:
    """Print the skipped/pending/diverged scan summary."""
    print(
        f"{len(scan_result.skipped)} skipped (complete), "
        f"{len(scan_result.pending)} pending, "
        f"{len(scan_result.diverged)} diverged"
    )


def register_atexit_summary(pending_count: int, session_start: float) -> None:
    """Register belt-and-braces atexit summary for SIGTERM'd sessions."""

    def print_atexit_summary() -> None:
        elapsed_hours = (time.time() - session_start) / 3600.0
        print(
            f"SESSION SUMMARY (atexit): {pending_count} pending at start, "
            f"{elapsed_hours:.2f} hours elapsed"
        )

    atexit.register(print_atexit_summary)


def run_pending(
    matrix: tuple[RunSpec, ...],
    base_dir: str | Path,
    run_fn: Callable[[RunSpec, Path], dict[str, Any]],
    scan_fn: Callable[[tuple[RunSpec, ...], str | Path], ScanResult] = scan_runs,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
) -> dict[str, Any]:
    """Scan, report, execute pending runs, register atexit handler."""
    base = Path(base_dir)
    session_start = time.time()

    scan_result = scan_fn(matrix, base)

    announce_scan_results(scan_result)

    register_atexit_summary(len(scan_result.pending), session_start)

    results, failed = flush_pending_runs(
        scan_result.pending, base, run_fn, session_start, dir_name_fn
    )

    return {
        "completed": len(results),
        "skipped": len(scan_result.skipped),
        "diverged": len(scan_result.diverged),
        "failed": failed,
        "results": results,
    }

In [ ]:
%%writefile src/sjfpo/train_sjfpo.py
"""SJFPO online training entry (LeRobot-native, MuJoCo-only).

``generate_rollout`` gathers a batch, ``forward`` computes the loss; this loop
only steps the optimizer, evaluates, and checkpoints.
"""

from __future__ import annotations

import dataclasses
import json
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import torch
from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.utils.random_utils import set_seed

from sjfpo.checkpointing import load_policy_checkpoint, save_policy_checkpoint
from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.evaluator import Evaluator, check_divergence, mean_chunk_jerk, render_episode_frames
from sjfpo.modeling_sjfpo import SJFPOPolicy
from sjfpo.utils import (
    maybe_compile,
    resolve_device,
    to_numpy_1d,
)
from sjfpo.wandb_logging import (
    finish_run,
    init_wandb_run_for_method,
    log_eval_summary,
    log_eval_video,
    log_metrics,
)


class _EvalPolicy:
    """Single-action eval adapter over ``SJFPOPolicy.select_action``."""

    def __init__(self, policy: SJFPOPolicy):
        self.policy = policy

    def reset(self) -> None:
        self.policy.reset()

    def __call__(self, obs: np.ndarray) -> np.ndarray:
        action = self.policy.select_action({"observation.state": obs})
        return to_numpy_1d(action)  # atleast_1d: 1-D envs must stay [1], not scalar


def build_policy(
    obs_dim: int, action_dim: int, cfg: SJFPOConfig, device=None,
    action_low: np.ndarray | None = None,
    action_high: np.ndarray | None = None,
) -> SJFPOPolicy:
    """Attach runtime observation/action features, then build the policy."""
    device = torch.device(device) if device is not None else resolve_device()
    policy_cfg = dataclasses.replace(
        cfg,
        input_features={
            "observation.state": PolicyFeature(FeatureType.STATE, (obs_dim,))
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (action_dim,))},
        n_action_steps=max(1, round(cfg.executed_vs_predicted_ratio * cfg.action_horizon)),
    )
    policy = SJFPOPolicy(policy_cfg).to(device)
    # Spectral is parameter-free, normalizer is buffers-only; the snapshot
    # deep-copies the compiled actor so l_old stays compiled too.
    policy.actor = maybe_compile(policy.actor)
    policy.old_actor = maybe_compile(policy.old_actor)
    policy.critic = maybe_compile(policy.critic)
    if action_low is not None and action_high is not None:
        policy.set_action_bounds(action_low, action_high)
    return policy


def build_optimizer(policy: SJFPOPolicy, cfg: SJFPOConfig):
    """Lerobot preset AdamW; bitsandbytes 8-bit AdamW when opted in."""
    params = policy.get_optim_params()["params"]
    if cfg.optimizer_8bit:
        try:
            import bitsandbytes as bnb
        except ImportError as e:
            raise RuntimeError(
                "optimizer_8bit=True needs bitsandbytes (pip install bitsandbytes)"
            ) from e
        return bnb.optim.AdamW8bit(
            params, lr=cfg.optimizer_lr, weight_decay=cfg.optimizer_weight_decay
        )
    kwargs = dataclasses.asdict(cfg.get_optimizer_preset())
    kwargs.pop("grad_clip_norm", None)  # applied manually after backward
    if next(policy.parameters()).is_cuda:
        kwargs["fused"] = True  # single-kernel AdamW step; CPU falls back
    return torch.optim.AdamW(params, **kwargs)


def sample_minibatch(
    batch: dict[str, torch.Tensor], indices: torch.Tensor
) -> dict[str, torch.Tensor]:
    """Slice dim-0 rows; every batch value is a [B, ...] tensor."""
    return {k: v[indices] for k, v in batch.items()}


def random_baseline(evaluator: Evaluator, env: Any) -> float:
    """Mean random-policy return for divergence gating (1.0 if empty/zero)."""
    results = evaluator.evaluate(lambda obs: env.action_space.sample(), out_dir=None)
    r_rand = float(np.mean([r["return"] for r in results])) if results else 1.0
    return r_rand or 1.0


def train_sjfpo_run(
    env: Any,
    out_dir: str | Path,
    cfg: SJFPOConfig | None = None,
    *,
    seed: int = 3407,
    total_steps: int = 2000,
    eval_steps: int = 1000,
    save_steps: int = 1000,
    obs_dim: int | None = None,
    action_dim: int | None = None,
    resume_from_checkpoint: str | Path | None = None,
) -> dict[str, Any]:
    """Run a single SJFPO training run, returning summary with artifacts.

    Plain single-process PyTorch (``PYTHONPATH=src python scripts/train.py
    ...``). Mixed precision auto-detects: bf16 autocast matmuls when the CUDA
    device supports bf16 (see ``utils.is_bf16_available``), fp32 otherwise.

    ``resume_from_checkpoint`` points at a ``save_policy_checkpoint`` directory
    (``checkpoint_<step>`` or ``checkpoint_final``): policy, optimizer, and
    counters continue from there. ``total_steps`` stays an absolute env-step
    budget (the run stops when ``global_step`` reaches it). The checkpoint's
    own config shapes the policy; CLI hyperparameter overrides do not apply
    to a resumed run (scripts/train.py warns about them).
    """
    cfg = cfg or SJFPOConfig()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    unwrapped = getattr(env, "unwrapped", env)
    spec = getattr(unwrapped, "spec", None)
    env_name = getattr(spec, "id", "unknown") if spec is not None else "unknown"
    if obs_dim is None or action_dim is None:
        obs_dim = int(np.prod(env.observation_space.shape))
        action_dim = int(np.prod(env.action_space.shape))
    set_seed(seed)
    device = resolve_device()  # SJFPO_DEVICE=cpu|auto honors the CPU override
    train_env = env
    if cfg.num_envs > 1 and env_name != "unknown":
        # Build BEFORE any CUDA init: fork-based workers inherit the parent's
        # CUDA context otherwise (CleanRL/SB3 order the same way). Keep
        # evaluation single-env; fixtures without a registered id stay local.
        import gymnasium as gym

        train_env = gym.vector.AsyncVectorEnv(
            [lambda: gym.make(env_name) for _ in range(cfg.num_envs)]
        )
    # Real env bounds (not ±1): InvertedPendulum ±3, Pusher ±2, Humanoid ±0.4.
    action_low = np.atleast_1d(np.asarray(env.action_space.low, dtype=np.float32))
    action_high = np.atleast_1d(np.asarray(env.action_space.high, dtype=np.float32))
    global_step = 0
    if resume_from_checkpoint is not None:
        # Exact continuation: weights, optimizer state, and counters come
        # from the checkpoint dir (raises ResumeError when incomplete).
        policy, optimizer, counters = load_policy_checkpoint(
            resume_from_checkpoint, device=device
        )
        global_step = int(counters["global_step"])
        policy.train()  # loader leaves eval mode; rollouts train like fresh
        (out_dir / "resume_info.json").write_text(
            json.dumps(
                {
                    "resume_from_checkpoint": str(resume_from_checkpoint),
                    "restored_global_step": global_step,
                },
                indent=2,
            ),
            encoding="utf-8",
        )
    else:
        policy = build_policy(obs_dim, action_dim, cfg, device=device,
                               action_low=action_low, action_high=action_high)
        optimizer = build_optimizer(policy, cfg)
    print(policy, file=sys.stderr)
    evaluator = Evaluator(env, eval_seed=10000 + seed, max_episodes=12)
    eval_policy = _EvalPolicy(policy)
    r_rand = random_baseline(evaluator, env)
    wandb_enabled = init_wandb_run_for_method(
        method="sjfpo",
        env_name=env_name,
        seed=seed,
        total_steps=total_steps,
        cfg_dict=dataclasses.asdict(cfg),
    )

    eval_count = 0
    consecutive_below = 0
    diverged = False
    best_return: float | None = None
    steps_to_best: int | None = None
    artifacts: list = []
    all_eval_returns: list = []
    t_start = time.time()
    try:
        while global_step < total_steps:
            steps_to_go = total_steps - global_step
            collect_steps = min(cfg.chunks_per_update * cfg.action_horizon, steps_to_go)
            batch = policy.generate_rollout(train_env, collect_steps,
                                              action_low=action_low, action_high=action_high)
            global_step += int((~batch["action_is_pad"]).sum())
            # Exploration smoothness: mean jerk over executed chunk prefixes.
            rollout_jerk = None
            if "action" in batch and "action_is_pad" in batch:
                rollout_jerk = mean_chunk_jerk(
                    batch["action"].detach().cpu().numpy(),
                    batch["action_is_pad"].detach().cpu().numpy(),
                )
            policy.train()
            infos = []
            grad_norms = []  # pre-clip totals; clip_grad_norm_ returns the norm
            # PPO-style: freeze the behavior snapshot once per collection round
            # (OFF semantics: fixed pi_old + stored eps/t for all epochs), then
            # sweep shuffled minibatches with reshuffled partitions per epoch.
            policy.snapshot_behavior_policy()
            n_chunks = batch["advantages"].shape[0]
            n_mb = max(1, min(cfg.num_minibatches, n_chunks))
            for _ in range(cfg.num_epochs):
                for mb_idx in torch.randperm(n_chunks).split(max(1, n_chunks // n_mb)):
                    optimizer.zero_grad()
                    mb_idx = mb_idx.to(batch["advantages"].device)
                    loss, epoch_info = policy(sample_minibatch(batch, mb_idx))
                    loss.backward()
                    all_params = [
                        p for group in optimizer.param_groups for p in group["params"]
                    ]
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        all_params, cfg.grad_clip_norm
                    )
                    grad_norms.append(float(grad_norm))
                    optimizer.step()
                    policy.step_count += 1
                    infos.append(epoch_info)
            info = {k: sum(d[k] for d in infos) / len(infos) for k in infos[0]}
            grad_norm_mean = sum(grad_norms) / len(grad_norms) if grad_norms else 0.0
            if wandb_enabled:
                data = {f"train/{k}": v for k, v in info.items()}
                data["train/grad_norm_preclip"] = grad_norm_mean
                if rollout_jerk is not None:
                    data["train/rollout_jerk"] = rollout_jerk
                data["train/total_loss"] = (
                    info["policy_loss"] + cfg.vf_coef * info["value_loss"]
                )
                data.update(
                    {
                        "gae/advantages_mean": float(batch["advantages"].mean()),
                        "gae/advantages_std": float(batch["advantages"].std()),
                        "gae/returns_mean": float(batch["returns"].mean()),
                        "gae/values_mean": float(batch["values"].mean()),
                    }
                )
                log_metrics(data, step=global_step)
            diverged_now = False
            diverge_msg = ""
            if global_step % eval_steps < cfg.action_horizon or global_step >= total_steps:
                eval_results = evaluator.evaluate(eval_policy, out_dir=out_dir)
                mean_return = float(np.mean([r["return"] for r in eval_results]))
                all_eval_returns.append(mean_return)
                eval_count += 1
                if best_return is None or mean_return > best_return:
                    best_return = mean_return
                    steps_to_best = global_step
                if wandb_enabled:
                    log_eval_summary(
                        mean_return,
                        eval_results,
                        all_eval_returns,
                        eval_count,
                        global_step,
                        r_rand=r_rand,
                        consecutive_below=consecutive_below,
                        best_return=best_return,
                        steps_to_best=steps_to_best,
                    )
                    try:
                        frames, demo_return = render_episode_frames(
                            env_name, eval_policy, seed=10000 + seed
                        )
                        log_eval_video(frames, global_step)
                        log_metrics(
                            {
                                "eval/demo_return": demo_return,
                                "eval/video_ok": 1,
                            },
                            step=global_step,
                        )
                    except Exception:
                        log_metrics({"eval/video_ok": 0}, step=global_step)
                        pass  # headless/GL failures must never break training
                consecutive_below, diverged_now = check_divergence(
                    mean_return, r_rand, consecutive_below
                )
                if diverged_now:
                    diverge_msg = (
                        f"DIVERGED at step {global_step}: mean_return={mean_return:.2f} "
                        f"< 0.10*R_rand={0.10 * r_rand:.2f}"
                    )
            if diverged_now:
                diverged = True
                if wandb_enabled:
                    log_metrics(
                        {"eval/diverged": 1, "eval/diverge_step": global_step},
                        step=global_step,
                    )
                print(diverge_msg)
                break
            if global_step % save_steps < cfg.action_horizon or global_step >= total_steps:
                ckpt_dir = out_dir / f"checkpoint_{global_step}"
                save_policy_checkpoint(
                    ckpt_dir,
                    policy=policy,
                    optimizer=optimizer,
                    global_step=global_step,
                )
                artifacts.append(str(ckpt_dir / "model.safetensors"))
        elapsed = time.time() - t_start
        fps = global_step / max(elapsed, 1e-6)
        if wandb_enabled:
            log_metrics(
                {"timing/fps": fps, "timing/elapsed_s": elapsed}, step=global_step
            )
    finally:
        if train_env is not env:
            train_env.close()
        if wandb_enabled:
            finish_run(exit_code=1 if diverged else 0)

    ckpt_dir = out_dir / "checkpoint_final"
    save_policy_checkpoint(
        ckpt_dir,
        policy=policy,
        optimizer=optimizer,
        global_step=global_step,
    )
    artifacts.append(str(ckpt_dir / "model.safetensors"))
    csv_path = out_dir / "eval_history.csv"
    if csv_path.exists():
        artifacts.append(str(csv_path))
    return {
        "method": "sjfpo",
        "env_name": env_name,
        "seed": seed,
        "reward_mean_last10": (
            float(np.mean(all_eval_returns[-10:])) if all_eval_returns else 0.0
        ),
        "fps": fps,
        "artifacts": artifacts,
        "diverged": diverged,
        "global_step": global_step,
    }

In [ ]:
%%writefile src/sjfpo/utils.py
"""Shared stateless utilities (no sjfpo.* imports — circular-import prevention)."""

from __future__ import annotations

import contextlib
import os
import sys
from pathlib import Path

import numpy as np
import torch
from lerobot.processor.converters import from_tensor_to_numpy, to_tensor


def resolve_device() -> torch.device:
    """CUDA when available, else CPU (``SJFPO_DEVICE`` overrides)."""
    want = os.environ.get("SJFPO_DEVICE", "auto").lower()
    if want == "cpu":
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def is_bf16_available() -> bool:
    """True when the CUDA device supports bf16 matmuls (else fp32 path)."""
    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()


def filter_low_freq(x: torch.Tensor, k: int, dim: int = 1) -> torch.Tensor:
    """Keep only the ``k`` lowest-frequency bins along ``dim`` (action only)."""
    s = [slice(None)] * x.dim()
    s[dim] = slice(None, k)
    return x[tuple(s)]


def resolve_compute_dtype() -> torch.dtype:
    """Autocast matmul dtype (storage stays fp32; ``SJFPO_DTYPE`` overrides)."""
    want = os.environ.get("SJFPO_DTYPE", "auto").lower()
    if want == "fp32":
        return torch.float32
    if want == "bf16":
        return torch.bfloat16 if is_bf16_available() else torch.float32
    return torch.bfloat16 if is_bf16_available() else torch.float32


def autocast_context(device=None):
    """CUDA bf16 autocast when enabled and supported; else no-op."""
    if resolve_compute_dtype() != torch.bfloat16:
        return contextlib.nullcontext()
    try:
        dev_type = device.type if isinstance(device, torch.device) else None
    except Exception:
        dev_type = None
    if dev_type is None:
        dev_type = "cuda" if torch.cuda.is_available() else "cpu"
    if dev_type != "cuda" or not is_bf16_available():
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16)


def maybe_compile(module: torch.nn.Module) -> torch.nn.Module:
    """Compile an nn.Module with inductor after build, before train/infer.

    On by default; ``SJFPO_COMPILE=0`` opts out (e.g. driver-only CUDA
    without a toolkit, where inductor fails, or CPU-only debugging).
    CPU inductor hits a guard bug (``___from_numpy`` ADInplaceOrView
    DispatchKeySet mismatch on Colab) so it auto-disables when CUDA is
    unavailable — eager is numerically identical.
    Default dynamic-shape mode: batch shapes vary, and static-shape
    cudagraphs would waste the 8GB budget. Wrapping is lazy — the compile
    cost lands on the first train/infer call, not here. Params are untouched;
    checkpointing unwraps wrappers before saving, so artifacts hold plain
    eager keys whatever this flag was. Raises loudly (never silent
    fallback) when inductor itself fails — set ``SJFPO_COMPILE=0``.
    """
    if os.environ.get("SJFPO_COMPILE", "1") == "0":
        return module
    if not torch.cuda.is_available():
        return module
    # Guarded scalar raises (e.g. Critic's executed-step check) stay in the
    # compiled graph instead of forcing an eager fallback for the whole module.
    torch._dynamo.config.capture_scalar_outputs = True
    return torch.compile(module)


def detect_runtime_env() -> str:
    """``kaggle`` | ``colab`` | ``local`` (env vars win over path probes)."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def default_runs_base(env: str | None = None) -> Path:
    """Per-run sweep directory (kaggle / colab-drive-or-content / local .omo)."""
    env = env or detect_runtime_env()
    if env == "kaggle":
        return Path("/kaggle/working/runs")
    if env == "colab":
        drive = Path("/content/drive/MyDrive/spectral-joint-policy/runs")
        return drive if Path("/content/drive").exists() else Path("/content/runs")
    return Path(".omo/evidence/runs")


def default_analysis_output(env: str | None = None) -> Path:
    """Analysis output directory (mirrors default_runs_base)."""
    env = env or detect_runtime_env()
    if env == "kaggle":
        return Path("/kaggle/working/analysis")
    if env == "colab":
        drive = Path("/content/drive/MyDrive/spectral-joint-policy/analysis")
        return drive if Path("/content/drive").exists() else Path("/content/analysis")
    return Path(".omo/evidence/analysis")


def default_smoke_dir(env: str | None = None) -> Path:
    """Smoke-gate output directory (mirrors default_runs_base)."""
    env = env or detect_runtime_env()
    if env == "kaggle":
        return Path("/kaggle/working/smoke")
    if env == "colab":
        drive = Path("/content/drive/MyDrive/spectral-joint-policy/smoke")
        return drive if Path("/content/drive").exists() else Path("/content/smoke")
    return Path(".omo/evidence/smoke")


def to_device_async(x: torch.Tensor, device) -> torch.Tensor:
    """Host-to-device move that overlaps with CPU work on CUDA.

    Pinned staging + ``non_blocking=True`` lets DMA run while the collector
    stacks the next tensors; ordering stays safe on the default stream.
    Plain ``.to()`` on CPU.
    """
    device = torch.device(device)
    if device.type == "cuda" and x.device != device:
        return x.pin_memory().to(device, non_blocking=True)
    return x.to(device)


def to_numpy_1d(x: torch.Tensor | np.ndarray | float) -> np.ndarray:
    """to numpy with atleast_1d: singletons stay [1], never 0-d/scalar."""
    return np.atleast_1d(from_tensor_to_numpy(x))


def to_tensor_1d(
    x: torch.Tensor | np.ndarray | float,
    dtype: torch.dtype | None = torch.float32,
    device: torch.device | str | None = None,
) -> torch.Tensor:
    """to tensor with atleast_1d: singletons stay [1], never 0-d."""
    return torch.atleast_1d(to_tensor(x, dtype=dtype, device=device))

In [ ]:
%%writefile src/sjfpo/wandb_logging.py
"""wandb logging — sole wandb importer, enabled by default.

REQUIRED dependency (missing package = hard ImportError). Only off switch:
``SJFPO_DISABLE_WANDB=1``. Failed init fails fast, never degrades silently.
"""

from __future__ import annotations

import os

import numpy as np

import wandb


def wandb_disabled() -> bool:
    """Return True only when the single off switch is set."""
    return os.environ.get("SJFPO_DISABLE_WANDB") == "1"


def compact_value(v) -> str:
    """Short wandb-safe rendering: ints plain, floats via ``:g``."""
    if isinstance(v, bool):
        return str(int(v))
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:g}"
    return str(v)


def build_run_name(method: str, env_name: str, seed: int, cfg_dict: dict) -> str:
    """``{method}_{env}_s{seed}_{tags}`` from major config keys.

    Fixed tag order, missing keys skipped, so sjfpo/ppo/sac dicts all work:
    e.g. ``sjfpo_Hopper-v5_s3407_top0.3_d128x8_ep16_lr0.0001_eps0.05_mc16``.
    """
    tags: list[str] = []
    get = cfg_dict.get
    if get("top_p_spectral_tokens") is not None:
        tags.append(f"top{compact_value(get('top_p_spectral_tokens'))}")
    if get("hidden_dim") is not None and get("dit_depth") is not None:
        tags.append(f"d{get('hidden_dim')}x{get('dit_depth')}")
    epochs = get("num_epochs", get("learning_epochs"))
    if epochs is not None:
        tags.append(f"ep{epochs}")
    lr = get("optimizer_lr", get("lr"))
    if lr is not None:
        tags.append(f"lr{compact_value(lr)}")
    eps = get("epsilon_clip", get("ratio_clip"))
    if eps is not None:
        tags.append(f"eps{compact_value(eps)}")
    if get("use_per_bin_clip"):
        tags.append(
            f"bin{compact_value(get('epsilon_clip_lowfreq'))}"
            f"-{compact_value(get('epsilon_clip_highfreq'))}"
        )
    if get("num_mc_samples") is not None:
        tags.append(f"mc{get('num_mc_samples')}")
    base = f"{method}_{env_name}_s{seed}"
    return base if not tags else f"{base}_{'_'.join(tags)}"


def init_wandb_run_for_method(
    *,
    method: str,
    env_name: str,
    seed: int,
    total_steps: int,
    cfg_dict: dict,
) -> bool:
    """Init a run in the per-algo project (``method``); False when disabled."""
    if wandb_disabled():
        return False
    wandb.init(
        project=method,
        group=f"{method}_{env_name}",
        name=build_run_name(method, env_name, seed, cfg_dict),
        tags=[method, "mujoco", "cpu-first"],
        config={**cfg_dict, "method": method, "seed": seed, "env": env_name},
        job_type="smoke" if total_steps <= 10000 else "train",
        mode=os.environ.get("WANDB_MODE", "online"),
        reinit="finish_previous",
        force=True,
    )
    return True


def log_metrics(data: dict, step: int) -> None:
    """Log a dict at a global step; no-op when disabled."""
    if wandb_disabled():
        return
    wandb.log(data, step=step)


def log_eval_summary(
    mean_return: float,
    eval_results: list[dict],
    all_eval_returns: list[float],
    eval_count: int,
    global_step: int,
    *,
    r_rand: float | None = None,
    consecutive_below: int | None = None,
    best_return: float | None = None,
    steps_to_best: int | None = None,
) -> None:
    """Eval metrics + per-episode table in one commit (shared runner/baselines home)."""
    if wandb_disabled():
        return
    returns = [r["return"] for r in eval_results]
    eval_lengths = [r["length"] for r in eval_results]
    successes = [r["success"] for r in eval_results if r.get("success") is not None]
    jerks = [r["jerk"] for r in eval_results if r.get("jerk") is not None]
    data: dict = {
        "eval/mean_return": mean_return,
        "eval/min_return": float(np.min(returns)) if returns else 0.0,
        "eval/max_return": float(np.max(returns)) if returns else 0.0,
        "eval/std_return": float(np.std(returns)) if returns else 0.0,
        "eval/mean_length": float(np.mean(eval_lengths)) if eval_lengths else 0.0,
        "eval/max_length": float(np.max(eval_lengths)) if eval_lengths else 0.0,
        "eval/std_length": float(np.std(eval_lengths)) if eval_lengths else 0.0,
        "eval/episode_count": eval_count,
    }
    if successes:
        data["eval/success_rate"] = float(np.mean(successes))
    if jerks:
        data["eval/mean_jerk"] = float(np.mean(jerks))
        data["eval/max_jerk"] = float(np.max(jerks))
    if best_return is not None:
        data["eval/best_mean_return"] = float(best_return)
    if steps_to_best is not None:
        data["eval/steps_to_best"] = int(steps_to_best)
    if r_rand is not None:
        data["eval/random_baseline"] = float(r_rand)
    if consecutive_below is not None:
        data["eval/consecutive_below"] = int(consecutive_below)
    if eval_results:
        rows = [
            (r["episode"], r["return"], r["length"], r.get("success"), r.get("jerk"))
            for r in eval_results
        ]
        data["eval/episodes"] = wandb.Table(
            columns=["episode", "return", "length", "success", "jerk"],
            data=[list(r) for r in rows],
        )
    if len(all_eval_returns) >= 2:
        prev, curr = all_eval_returns[-2], all_eval_returns[-1]
        data["eval/reward_improvement"] = curr - prev
        data["eval/reward_improvement_pct"] = (curr - prev) / max(abs(prev), 1e-8)
    log_metrics(data, step=global_step)


def log_eval_video(frames: list, global_step: int, *, fps: int = 30) -> None:
    """One eval episode as wandb.Video; no-op when disabled/empty/unrenderable."""
    if wandb_disabled() or not frames:
        return
    try:
        import numpy as _np

        video = _np.stack([_np.asarray(f, dtype=_np.uint8) for f in frames])
        if video.ndim != 4:
            return
        log_metrics({"eval/video": wandb.Video(video, fps=fps, format="mp4")}, step=global_step)
    except Exception:
        pass


def finish_run(exit_code: int = 0) -> None:
    """Finish the run; no-op when disabled."""
    if wandb_disabled():
        return
    wandb.finish(exit_code=exit_code)

In [ ]:
%%writefile scripts/train.py
"""Single-run training CLI for SJFPO/PPO/SAC (MuJoCo-only, CUDA-default, fp32).

One run per invocation; draccus builds the run config (singular values only).
Sweeps live in scripts/sweep.sh: one call here per method x env x seed.

    python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 2000
    python scripts/train.py --method sjfpo --env Hopper-v5 --sjfpo.optimizer_lr 3e-4
    python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 \
        --resume .omo/evidence/train/sjfpo_Hopper-v5_seed0/checkpoint_1000 \
        --total_steps 4000
    METHODS="sjfpo ppo" ENVS=Hopper-v5 SEEDS="0 1" scripts/sweep.sh

Flags use underscores (draccus convention, same as lerobot-train).

Precedence: CLI flags > --config YAML (flat, legacy) > dataclass defaults.
The flat YAML only fills method fields the CLI left at defaults. Tuple/dict
fields (hidden, ...) are YAML-only (draccus takes one token each).

Exit codes: 0 success, 1 usage error, 2 runtime/divergence abort,
4 required artifact missing (draccus parse errors exit 2).
"""

from __future__ import annotations

import dataclasses
import json
import math
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import draccus
import torch  # noqa: E402
torch.backends.fp32_precision = "ieee"
torch.backends.cuda.matmul.fp32_precision = "ieee"
torch.backends.cudnn.fp32_precision = "ieee"
torch.backends.cudnn.conv.fp32_precision = "tf32"
torch.backends.cudnn.rnn.fp32_precision = "tf32"
# Ensure src/ is importable
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from sjfpo.config import (  # noqa: E402
    KNOWN_ENVS,
    PPOConfig,
    SACConfig,
    lookup_env_dims,
)
from sjfpo.configuration_sjfpo import SJFPOConfig  # noqa: E402

VALID_ENVS = KNOWN_ENVS


@dataclass
class TrainConfig:
    """One training run (singular values only; sweep axes live in sweep.sh)."""

    method: Literal["sjfpo", "ppo", "sac"] = "sjfpo"
    env: str | None = None
    seed: int = 3407
    total_steps: int | None = None
    eval_steps: int | None = None
    save_steps: int | None = None
    obs_dim: int | None = None
    action_dim: int | None = None
    out_dir: str = ".omo/evidence/train"
    config: str | None = None  # legacy flat YAML overlay for the method config
    # Resume an sjfpo checkpoint dir (checkpoint_<step>/ or checkpoint_final/):
    # continues policy + optimizer + counters into a FRESH run dir
    # (<base>_resume[/2...]). total_steps stays an absolute env-step budget.
    resume: str | None = None
    sjfpo: SJFPOConfig = field(default_factory=SJFPOConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    sac: SACConfig = field(default_factory=SACConfig)


def load_yaml_dict(path: str | Path) -> dict:
    """Load YAML via safe_load only; friendly error when pyyaml is missing."""
    try:
        import yaml
    except ImportError as exc:
        raise SystemExit(
            "ERROR: pyyaml is required for --config (pip install pyyaml)"
        ) from exc
    with open(path, encoding="utf-8") as fh:
        data = yaml.safe_load(fh)
    if data is None:
        return {}
    if not isinstance(data, dict):
        raise SystemExit(f"ERROR: config {path} must be a YAML mapping")
    return data


def coerce_value(cfg_cls: type, raw: dict) -> dict:
    fields = {f.name for f in dataclasses.fields(cfg_cls)}
    out = {k: v for k, v in raw.items() if k in fields}
    for key in ("hidden",):
        if isinstance(out.get(key), list):
            out[key] = tuple(out[key])
    return out


def apply_flat_yaml(method_cfg, yaml_dict: dict):
    """Legacy flat --config YAML fills method fields the CLI left at defaults."""
    raw = coerce_value(type(method_cfg), yaml_dict)
    if not raw:
        return method_cfg
    fresh = type(method_cfg)()
    cli_touched = {
        f.name
        for f in dataclasses.fields(method_cfg)
        if getattr(method_cfg, f.name) != getattr(fresh, f.name)
    }
    overrides = {k: v for k, v in raw.items() if k not in cli_touched}
    return dataclasses.replace(method_cfg, **overrides) if overrides else method_cfg


def select_method_cfg(args: TrainConfig, yaml_dict: dict):
    """Nested draccus sub-config for the method, overlaid with flat YAML."""
    method_cfg = {"sjfpo": args.sjfpo, "ppo": args.ppo, "sac": args.sac}[args.method]
    return apply_flat_yaml(method_cfg, yaml_dict)


def resumed_cfg_overrides(cfg) -> list[str]:
    """Scalar fields differing from dataclass defaults.

    Used to warn with --resume: the checkpoint config wins there, so CLI
    overrides of these fields are silently ignored. Only plain scalars are
    compared (dict/tensor features have no meaningful CLI equality).
    """
    try:
        fresh = type(cfg)()
    except Exception:
        return []
    touched = []
    for f in dataclasses.fields(cfg):
        try:
            cur, default = getattr(cfg, f.name), getattr(fresh, f.name)
        except Exception:
            continue
        if isinstance(cur, (bool, int, float, str)) or cur is None:
            if isinstance(default, (bool, int, float, str)) or default is None:
                if cur != default:
                    touched.append(f.name)
    return sorted(touched)


def resolve_run_dir(out_base: str | Path, method: str, env: str, seed: int, resume: str | None) -> Path:
    """Run dir for one invocation; resumed runs get a fresh suffixed dir.

    ``<method>_<env>_seed<seed>`` normally, else the first free
    ``<base>_resume``, ``<base>_resume2``, ... so a resume never overwrites
    the original run (or an earlier resume).
    """
    base = Path(out_base) / f"{method}_{env}_seed{seed}"
    if not resume:
        return base
    candidate = Path(str(base) + "_resume")
    i = 2
    while candidate.exists():
        candidate = Path(f"{base}_resume{i}")
        i += 1
    return candidate


def run_single(
    method: str,
    env_id: str,
    seed: int,
    out_dir: Path,
    cfg,
    total_steps: int,
    eval_steps: int,
    save_steps: int,
    obs_dim: int | None,
    action_dim: int | None,
    n_obs_steps: int = 16,
    resume_from_checkpoint: str | None = None,
) -> dict:
    """Dispatch one run to train_sjfpo_run/ppo_train_run/sac_train_run."""
    import gymnasium as gym

    if env_id.startswith("AdroitHand"):
        try:
            import gymnasium_robotics  # noqa: F401 (registers Adroit env ids)
        except ImportError:
            raise RuntimeError(
                f"{env_id} needs gymnasium-robotics (pip install gymnasium-robotics)"
            )
    env = gym.make(env_id)
    try:
        # Registry fills unset dims; the live space is ground truth on mismatch.
        reg_obs, reg_act = lookup_env_dims(env_id)
        live_obs = int(math.prod(env.observation_space.shape))
        live_act = int(math.prod(env.action_space.shape))
        if (reg_obs, reg_act) != (live_obs, live_act):
            raise ValueError(
                f"registry dims {(reg_obs, reg_act)} != live {env_id} dims "
                f"{(live_obs, live_act)}; update configs/envs.yaml"
            )
        obs_dim = reg_obs if obs_dim is None else obs_dim
        action_dim = reg_act if action_dim is None else action_dim
        if (obs_dim, action_dim) != (live_obs, live_act):
            raise ValueError(
                f"requested dims {(obs_dim, action_dim)} != live {env_id} dims "
                f"{(live_obs, live_act)}"
            )
        from sjfpo.baselines import ppo_train_run, sac_train_run
        from sjfpo.train_sjfpo import train_sjfpo_run

        runners = {
            "sjfpo": (train_sjfpo_run, {"obs_dim": obs_dim, "action_dim": action_dim, "resume_from_checkpoint": resume_from_checkpoint}),
            "ppo": (ppo_train_run, {"n_obs_steps": n_obs_steps}),
            "sac": (sac_train_run, {"n_obs_steps": n_obs_steps}),
        }
        if resume_from_checkpoint is not None and method != "sjfpo":
            raise ValueError(f"--resume is sjfpo-only, got method={method!r}")
        run_fn, extra = runners[method]
        return run_fn(
            env=env,
            out_dir=out_dir,
            cfg=cfg,
            seed=seed,
            total_steps=total_steps,
            eval_steps=eval_steps,
            save_steps=save_steps,
            **extra,
        )
    finally:
        env.close()


def main(argv: list[str] | None = None) -> int:
    # Tensor-core fp32 matmuls (TF32) for leftover fp32 GEMMs; autocast still
    # handles bf16. Precision impact is far below the numeric gates.
    torch.set_float32_matmul_precision("high")
    args = draccus.parse(config_class=TrainConfig, args=argv)
    if not args.env:
        print("ERROR: --env is required", file=sys.stderr)
        return 1
    if args.env not in VALID_ENVS:
        print(
            f"ERROR: unknown env {args.env!r}; choices: {', '.join(VALID_ENVS)}",
            file=sys.stderr,
        )
        return 1
    yaml_dict: dict = load_yaml_dict(args.config) if args.config else {}
    total = args.total_steps or int(yaml_dict.get("total_steps", 2000))
    eval_s = args.eval_steps or int(yaml_dict.get("eval_steps", min(1000, total)))
    ckpt_i = args.save_steps or int(yaml_dict.get("save_steps", min(1000, total)))
    n_obs_steps = int(yaml_dict.get("n_obs_steps", 16))
    cfg = select_method_cfg(args, yaml_dict)
    if args.resume and args.method != "sjfpo":
        print(f"ERROR: --resume is sjfpo-only, got --method {args.method!r}", file=sys.stderr)
        return 1
    if args.resume and not Path(args.resume).is_dir():
        print(f"ERROR: --resume dir not found: {args.resume}", file=sys.stderr)
        return 1
    if args.resume:
        overridden = resumed_cfg_overrides(cfg)
        if overridden:
            print(
                "WARNING: --resume restores the checkpoint config; "
                f"CLI overrides ignored for: {', '.join(overridden)}",
                file=sys.stderr,
            )
    # Same layout the old matrix used (sweep.format_run_directory_name),
    # suffixed when resuming so the source run is never overwritten.
    run_dir = resolve_run_dir(args.out_dir, args.method, args.env, args.seed, args.resume)
    run_dir.mkdir(parents=True, exist_ok=True)
    try:
        result = run_single(
            args.method,
            args.env,
            args.seed,
            run_dir,
            cfg,
            total,
            eval_s,
            ckpt_i,
            args.obs_dim,
            args.action_dim,
            n_obs_steps,
            args.resume,
        )
    except Exception as exc:
        print(
            json.dumps(
                {
                    "method": args.method,
                    "env": args.env,
                    "seed": args.seed,
                    "error": str(exc),
                }
            )
        )
        return 2
    for artifact in result.get("artifacts", []):
        if not Path(artifact).exists():
            print(f"MISSING ARTIFACT: {artifact}", file=sys.stderr)
            return 4
    print(json.dumps(result))
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile configs/envs.yaml
# State-based MuJoCo prototype ladder. `train.py --env <id>` resolves
# obs_dim/action_dim here; --obs-dim/--act-dim still override, and the live
# env space is validated against the registry at startup (mismatch = loud
# error, usually a gym version change — re-measure and update).
# Dims below MEASURED with gymnasium 1.x + mujoco on 2026-09-14.
InvertedPendulum-v5: {obs_dim: 4, action_dim: 1, tier: smoke}
InvertedDoublePendulum-v5: {obs_dim: 9, action_dim: 1, tier: smoke}
Reacher-v5: {obs_dim: 10, action_dim: 2, tier: smoke}
Hopper-v5: {obs_dim: 11, action_dim: 3, tier: core}
HalfCheetah-v5: {obs_dim: 17, action_dim: 6, tier: core}
Walker2d-v5: {obs_dim: 17, action_dim: 6, tier: core}
Swimmer-v5: {obs_dim: 8, action_dim: 2, tier: core}
Pusher-v5: {obs_dim: 23, action_dim: 7, tier: hard}
Ant-v5: {obs_dim: 105, action_dim: 8, tier: hard}
Humanoid-v5: {obs_dim: 348, action_dim: 17, tier: hard}
AdroitHandDoor-v1: {obs_dim: 39, action_dim: 28, tier: dexterous, requires: gymnasium-robotics}
AdroitHandPen-v1: {obs_dim: 45, action_dim: 24, tier: dexterous, requires: gymnasium-robotics}
AdroitHandHammer-v1: {obs_dim: 46, action_dim: 26, tier: dexterous, requires: gymnasium-robotics}

In [ ]:
%%writefile configs/ppo.yaml
# skrl PPO baseline defaults — mirrors src/sjfpo/config.py:PPOConfig (frozen, skrl 2.1.0).
# CUDA-default with CPU fallback (resolve_device, SJFPO_DEVICE override). Frame-stack n_obs_steps=16 passthrough.
rollouts: 2048
learning_epochs: 10
mini_batches: 8
lr: 0.0003
kl_threshold: 0.01
grad_norm_clip: 1.0
discount: 0.99
gae_lambda: 0.95
ratio_clip: 0.2
value_loss_scale: 1.0
hidden: [256, 256, 256]

In [ ]:
%%writefile configs/sac.yaml
# skrl SAC baseline defaults — mirrors src/sjfpo/config.py:SACConfig (frozen, skrl 2.1.0).
# CUDA-default with CPU fallback (resolve_device, SJFPO_DEVICE override). Frame-stack n_obs_steps=16 passthrough.
lr: 0.0003
batch_size: 256
memory_size: 300000
gradient_steps: 1
polyak: 0.005
discount: 0.99
initial_random_steps: 5000
hidden: [256, 256, 256]

In [ ]:
%%writefile configs/shared.yaml
# Shared experiment defaults — mirrors src/sjfpo/config.py RUN_MATRIX axes + budgets.
# Source of truth: config.py RunSpec/RUN_MATRIX (45 runs: 3 methods x 3 envs x 5 seeds).
# MuJoCo-only (flat Box). fp32-only. CUDA-default with CPU fallback.
run_matrix:
  methods: [sjfpo, ppo, sac]
  envs: [HalfCheetah-v5, Walker2d-v5, Hopper-v5]
  seeds: [0, 1, 2, 3, 4]
  total_steps: 1000000
  eval_steps: 50000
  save_steps: 25000
tiny:
  env: Hopper-v5
  seed: 0
  steps:
    sjfpo: 20000
    ppo: 20000
    sac: 20000
  eval_steps: 10000
  save_steps: 10000
eval:
  eval_steps: 50000
  save_steps: 25000
  max_episodes: 12
output:
  out_dir: outputs/
# wandb: enabled by default; SJFPO_DISABLE_WANDB=1 is the only off switch (project = algo name at runtime).
# Never log secrets. See src/sjfpo/wandb_logging.py.

In [ ]:
%%writefile configs/sjfpo.yaml
# SJFPO defaults — mirrors src/sjfpo/configuration_sjfpo.py:SJFPOConfig.
# fp32 storage (bf16 autocast matmuls on CUDA; torch.compile on, SJFPO_COMPILE=0 off).
num_mc_samples: 16
num_euler_steps: 8
num_inference_steps: 16
chunks_per_update: 128
num_epochs: 10
optimizer_lr: 0.0003
optimizer_weight_decay: 0.0001
gamma: 0.999
gae_lambda: 0.995
epsilon_clip: 0.05
use_per_bin_clip: false
epsilon_clip_lowfreq: 0.02
epsilon_clip_highfreq: 0.10
cfm_clamp: 20.0
log_ratio_cap: 10.0
advantage_clamp: 100.0
use_clipped_value_loss: false
min_timestep: 0.005
max_timestep: 0.995
dit_depth: 4
dit_heads: 4
dit_mlp_ratio: 4.0
dit_time_dim: 128
timestep_scale: 1000.0
vf_coef: 0.01
grad_clip_norm: 1.0
n_obs_steps: 16
action_horizon: 16
executed_vs_predicted_ratio: 0.5
top_p_spectral_tokens: 1.0
hidden_dim: 128

In [ ]:
%%writefile requirements-linux.txt
absl-py==2.5.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
annotated-types==0.8.0
anyio==4.15.1
attrs==26.1.0
av==15.1.0
basedpyright==1.40.1
bitsandbytes==0.50.2
certifi==2026.7.22
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
cmake==4.1.3
colorama==0.4.6
contourpy==1.3.3
cuda-bindings==13.4.1
cuda-pathfinder==1.8.1
cuda-toolkit==13.0.2
cycler==0.12.1
datasets==4.8.5
dill==0.4.1
draccus==0.11.6
einops==0.8.2
einx==0.4.3
etils==1.14.0
farama-notifications==0.0.6
fastjsonschema==2.22.2
filelock==3.32.6
fonttools==4.64.0
frozendict==2.4.7
frozenlist==1.8.0
fsspec==2026.2.0
gitdb==4.0.12
gitpython==3.1.62
glfw==2.10.2
googleapis-common-protos==1.75.3
grpcio==1.83.1
gymnasium==1.3.0
gymnasium-robotics==1.4.2
h11==0.16.0
hf-xet==1.6.0
httpcore==1.0.9
httpx==0.28.1
huggingface-hub==1.31.0
idna==3.19
imageio==2.37.4
iniconfig==2.3.0
jax==0.10.0
jaxlib==0.10.0
jinja2==3.1.6
joblib==1.6.0
jsonlines==4.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-core==5.9.1
jupytext==1.19.5
kiwisolver==1.5.1
lerobot==0.6.1
markdown==3.10.3
markdown-it-py==4.2.0
markupsafe==3.0.3
matplotlib==3.11.1
mdit-py-plugins==0.6.1
mdurl==0.1.2
mergedeep==1.3.4
ml-dtypes==0.6.0
mpmath==1.3.0
mujoco==3.13.0
multidict==6.8.0
multiprocess==0.70.19
mypy-extensions==1.1.0
narwhals==2.26.0
nbformat==5.11.1
networkx==3.6.1
nodejs-wheel-binaries==24.19.0
numpy==2.2.6
nvidia-cublas==13.1.0.3
nvidia-cuda-cupti==13.0.85
nvidia-cuda-nvrtc==13.0.88
nvidia-cuda-runtime==13.0.96
nvidia-cudnn-cu13==9.19.0.56
nvidia-cufft==12.0.0.61
nvidia-cufile==1.15.1.6
nvidia-curand==10.4.0.35
nvidia-cusolver==12.0.4.66
nvidia-cusparse==12.6.3.3
nvidia-cusparselt-cu13==0.8.0
nvidia-nccl-cu13==2.28.9
nvidia-nvjitlink==13.0.88
nvidia-nvshmem-cu13==3.4.5
nvidia-nvtx==13.0.85
opencv-python-headless==4.13.0.92
opentelemetry-api==1.44.0
opentelemetry-exporter-otlp-proto-common==1.44.0
opentelemetry-exporter-otlp-proto-http==1.44.0
opentelemetry-proto==1.44.0
opentelemetry-sdk==1.44.0
opentelemetry-semantic-conventions==0.65b0
opt-einsum==3.4.0
packaging==25.0
pandas==2.3.3
pettingzoo==1.27.0
pillow==12.3.0
platformdirs==4.11.8
pluggy==1.6.0
propcache==0.5.2
protobuf==7.36.1
psutil==7.2.2
pyarrow==25.0.1
pydantic==2.13.5
pydantic-core==2.46.5
pygments==2.21.0
pyopengl==3.1.10
pyparsing==3.3.2
pytest==9.1.1
python-dateutil==2.9.0.post0
pytz==2026.3.post1
pyyaml==6.0.3
referencing==0.37.0
requests==2.34.2
rpds-py==2026.6.3
safetensors==0.8.0
scikit-learn==1.9.0
scipy==1.18.1
sentry-sdk==2.69.1
setuptools==81.0.0
six==1.17.0
skrl==2.1.0
smmap==5.0.3
sympy==1.14.0
tensorboard==2.21.0
tensorboard-data-server==0.7.2
termcolor==3.3.0
threadpoolctl==3.6.0
toml==0.10.2
torch==2.11.0
torchaudio==2.11.0
torchcodec==0.11.1
torchvision==0.26.0
tqdm==4.70.0
traitlets==5.16.1
triton==3.6.0
typing-extensions==4.16.0
typing-inspect==0.9.0
typing-inspection==0.4.4
tzdata==2026.4
urllib3==2.7.0
wandb==0.30.0
werkzeug==3.1.8
xxhash==4.0.1
yarl==1.24.5
zipp==4.1.0

In [ ]:
# Offline install from the attached wheels dataset (no internet on Kaggle).
# Edit WHEELS_DIR to the /kaggle/input/... path of your packages dataset.
WHEELS_DIR = "/kaggle/input/sjfpo-packages"
!pip install -q --no-index --find-links=$WHEELS_DIR -r requirements-linux.txt

In [ ]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

In [ ]:
# Smoke: 2k steps, single SJFPO run on Hopper (eval + checkpoints included).
!python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 2000

In [ ]:
# Full run: 1M steps with the matrix budgets (50k eval / 25k ckpt).
!python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 1000000 --eval_steps 50000 --save_steps 25000